# 24 - Conclusions and findings

This is the last notebook in the series. It does no new modeling: it re-reads the outputs that notebooks `00` through `23`
wrote under `outputs/nb/<stage>/`, and assembles them into one honest report on what the project found, what it failed
to find, and what the reader must not conclude from it.

**The question this notebook answers:** after building a national travel-adjacency graph of Israel's public transport and attacking it
from a dozen directions - centrality measures, structural cut nodes, substitutability, simulated disruption, regional and
socioeconomic equity, graph learning, the six transport modes separately, scheduled travel time, service by time of day, a
population-weighted demand proxy and an explicit rerouting model - what actually holds up?

Seven findings carry the project:

1. **Operational criticality and structural criticality are two different things.** The stations carrying the largest service
   volume aren't the stations whose removal breaks the network. The rank correlation between weighted degree (service volume) and
   betweenness / articulation-point status is far from 1, and the overlap at the top of the ranking is small. Planning for one
   doesn't protect against the other.
2. **The network is resilient to random failure and fragile under targeted attack** - the classic signature of a heavy-tailed
   network. The size of the gap, and the fact that it shrinks considerably once the largest component's share is normalized by the
   *surviving* stations rather than by the original station count, are both reported below.
3. **The socioeconomic result is a Simpson's paradox, not a simple story of inequality.** The sign of the relationship between
   per-capita service and the socioeconomic cluster at the national level isn't the sign you get within individual cities.
4. **The rail sub-network is a completely different creature**: smaller by about two orders of magnitude, close to a path graph,
   and therefore dominated by cut nodes - but with only a few dozen stations, its socioeconomic tests lack statistical power and
   don't reach significance.
5. **"The network" is really six networks that share almost nothing structurally.** Stages 14-16 build one graph per mode in the
   GTFS. The bus network is a near-perfect single connected component; the light-rail graph is a handful of disconnected paths and
   trees in which almost every station is a cut node. Averaging them into one "public transport network" hides both.
6. **Shortest paths by hop count are the wrong distance measure, and stage 18 measures by how much.** Once the edges carry median
   scheduled travel time, hop-optimal paths almost never coincide with time-optimal paths, and they cost a large median time
   penalty. The betweenness rankings under the two measures disagree far more than the sampling control does within the same
   measure, so this is a real signal and not noise - and it's a limitation of every stage numbered below 18.
7. **"Critical" isn't one property.** Stage 23 scores every station under eight independent lenses - structural, operational,
   spatial, demand-weighted, temporal and passenger-time-based - and measures how much they agree. The average pairwise rank
   correlation is low, structural and operational criticality share almost nothing in their top-50 lists, and most flagged
   stations are flagged by a single lens only. This is the project's central claim, and now it's quantitative.

Several sub-results are **negative** and are reported as such: node2vec's advantage in link prediction mostly disappears once you
remove leakage between the training and test sets, the "resilience improvement from adding the suggested edges" the model produced
is ~0%, the embedding-based critical-station classifier reaches an F1 of a few percent up to around the low tens of percent, and
none of the rail socioeconomic correlations survive correction for multiple comparisons.

## Input

Every number below is **read from disk**; nothing is hard-coded. The notebook reads, where present:

| Stage | Folder under `outputs/nb/` | Outputs used here |
|---|---|---|
| 00 | `00_setup_and_data` | `feed_manifest.json` |
| 01 | `01_data_preparation` | `data_cleaning_report.json`, `route_type_distribution.csv` |
| 02 | `02_graph_construction` | `graph_build_summary.json` |
| 03 | `03_descriptive_analysis` | `network_summary.json`, `articulation_points.csv`, `bridges.csv` |
| 04 | `04_centrality_analysis` | `stop_metrics.csv`, `centrality_correlation_spearman.csv`, `betweenness_stability.csv` |
| 05 | `05_critical_station_isolation` | `isolation_summary.csv`, `critical_isolation.csv` |
| 06 | `06_robustness_analysis` | `disruption_results.csv`, `auc_summary.csv`, `damage_snapshots.csv` |
| 07 | `07_regional_comparison` | `regional_summary.csv`, `regional_significance.csv` |
| 08 | `08_community_detection` | `community_detection_summary.json`, `community_summary.csv` |
| 09 | `09_socioeconomic_equity` | `socioeconomic_summary.json`, `socioeconomic_within_cluster_correlation.csv` |
| 10 | `10_network_model_comparison` | `network_model_comparison.csv`, `degree_distribution_summary.csv`, `small_world_ratios.csv` |
| 11 | `11_rail_network_analysis` | `summary.json`, `single_station_damage.csv`, `rail_station_metrics.csv` |
| 12 | `12_rail_socioeconomic` | `rail_socioeconomic_correlations.csv` |
| 13 | `13_embeddings_link_prediction` | `link_prediction_results.csv`, `link_prediction_hard_negatives.csv`, `critical_classifier_results.csv`, `resilience_improvement.csv` |
| 14 | `14_multimodal_inventory` | `tables/mode_inventory.csv`, `tables/mode_network_summary.csv`, `mode_comparison_summary.json` |
| 15 | `15_bus_network` | `bus_summary.json`, `tables/bus_station_metrics.csv` |
| 16 | `16_lightrail_and_minor_modes` | `tables/minor_modes_summary.csv`, `tables/lightrail_station_metrics.csv`, `lightrail_summary.json` |
| 17 | `17_multimodal_transfer_hubs` | `tables/transfer_hubs.csv`, `multimodal_summary.json` (optional - skipped when absent) |
| 18 | `18_travel_time_network` | `traveltime_summary.json`, `tables/edges_traveltime.csv`, `tables/path_comparison.csv`, `tables/betweenness_agreement.csv` |
| 19 | `19_time_of_day_graphs` | `tables/window_summary.csv` |
| 20 | `20_dynamic_resilience` | `dynamic_summary.json`, `tables/dynamic_resilience.csv`, `tables/fragility_summary.csv`, `tables/window_rank_agreement.csv`, `tables/betweenness_noise_floor.csv` |
| 21 | `21_demand_weighted_criticality` | `demand_summary.json`, `tables/demand_weighted_criticality.csv`, `tables/rank_agreement.csv` |
| 22 | `22_rerouting_model` | `rerouting_summary.json`, `tables/rerouting_results.csv` |
| 23 | `23_critical_station_lenses` | `tables/lens_agreement.csv`, `tables/critical_station_lenses.csv`, `tables/lens_coverage.csv`, `lens_synthesis_summary.json` |

Any stage that wasn't run is **skipped with a message**, and doesn't cause a crash. The availability audit in section 5 states
exactly which stages are missing. Stage 17 in particular is optional: if its transfer-hub tables don't exist, the multimodal lens
is simply missing from the stage 23 comparison, and the notebook says so instead of failing.

## Output

Everything is written under `outputs/nb/24_conclusions/`:

* `tables/stage_availability.csv` - which stages were found and what they contain.
* `tables/headline_findings.csv` - the cross-stage summary table (one row per headline number).
* `tables/operational_vs_structural_overlap.csv` - top-K overlap between the service ranking and the structural ranking.
* `figures/operational_vs_structural.png`, `figures/project_at_a_glance.png`.


## 1. Environment setup

Identical to every other notebook in the series, so this notebook can be run standalone and also in Google Colab. `_ensure(...)`
pip-installs only packages that are genuinely missing, `find_repo_root()` climbs upward from the working directory looking for the
GTFS folder and clones the repo in Colab if it isn't found there. It then fixes `REPO`, `DATA` and `OUT`. Every later cell depends
on these three paths, so this has to run first. This notebook never touches `DATA` at all - it only reads stage outputs - but the
constant is kept so setup stays exactly the same across the series.


In [ ]:
# --- Environment bootstrap (safe to re-run, works locally and on Google Colab) ---
import os, sys, subprocess
from pathlib import Path

def _ensure(*pkgs):
    """Install only the packages that are actually missing."""
    import importlib.util
    alias = {"scikit-learn": "sklearn", "python-louvain": "community",
             "python-bidi": "bidi", "node2vec": "node2vec"}
    missing = [p for p in pkgs
               if importlib.util.find_spec(alias.get(p, p.replace("-", "_"))) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

def find_repo_root():
    """Find the repo locally; on Colab, clone it."""
    here = Path(os.getcwd()).resolve()
    for cand in [here, *here.parents]:
        if (cand / "israel-public-transportation").is_dir():
            return cand
    target = Path("/content/israel-transit-network-resilience")
    if not target.exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/seanfourman/israel-transit-network-resilience.git",
                        str(target)], check=True)
    return target

REPO = find_repo_root()
os.chdir(REPO)
DATA = REPO / "israel-public-transportation"
OUT = REPO / "outputs" / "nb"
OUT.mkdir(parents=True, exist_ok=True)
print("Repo root:", REPO)

## 2. Libraries, stage folder, and tunable constants

Only `pandas`, `numpy`, `matplotlib` and `seaborn` are needed: this notebook joins CSV/JSON files and draws two figures, it never
rebuilds a graph, so `networkx` is deliberately not imported. Following the project convention, this stage holds exactly one output
folder, `outputs/nb/24_conclusions/`, with the sub-folders `tables/` and `figures/`.

The constants are collected in one place: `OVERLAP_KS` controls the ranking depths at which agreement between operational and
structural criticality is measured, `TOP_N` sets how many rows the illustrative tables show, and `FIG_DPI` the figure resolution.


In [ ]:
# --- Libraries and stage folders ------------------------------------------
_ensure('pandas', 'numpy', 'matplotlib', 'seaborn')

import json
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', font_scale=1.05)

STAGE = OUT / '24_conclusions'      # everything this notebook produces
TABLES = STAGE / 'tables'
FIGURES = STAGE / 'figures'
TABLES.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

# --- Tunable constants ----------------------------------------------------
FIG_DPI = 150                      # figure resolution
TOP_N = 20                         # rows shown in illustrative tables
OVERLAP_KS = [10, 50, 100, 500]    # ranking depths for the agreement measurement
CRITICAL_QUANTILE = 0.90           # "critical" = top 10% betweenness (as in notebooks 05 and 07)

print('reading stages from :', OUT)
print('this stage          :', STAGE)

## 3. Handling Hebrew labels

The station names and region names in the Israeli GTFS are in Hebrew, and two panels of the summary figure print them. Matplotlib
doesn't implement Unicode's bidirectional algorithm, so right-to-left text is drawn reversed. The cell below applies a one-time
monkey-patch to `matplotlib.text.Text.set_text` so that any string containing Hebrew characters is converted to display order using
`python-bidi`, and picks a font that has Hebrew glyphs (Arial on Windows, DejaVu Sans elsewhere). The operation is idempotent. All
other text in this notebook is in English, per the submission requirement.


In [ ]:
# Stop names are Hebrew. Matplotlib does not apply the Unicode bidi algorithm, so
# Hebrew labels render reversed. Patch it once, before drawing any figure.
_ensure("python-bidi")
import re
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.text as mtext
from bidi.algorithm import get_display

_HEBREW_RE = re.compile(r"[\u0590-\u05FF]")

def fix_he(text):
    """Return display-ordered text. Non-Hebrew is returned untouched."""
    if not isinstance(text, str) or not _HEBREW_RE.search(text):
        return text
    return get_display(text)

def install_hebrew():
    # Arial exists on Windows; DejaVu Sans ships with matplotlib and covers Hebrew.
    matplotlib.rcParams["font.family"] = ["Arial", "DejaVu Sans"]
    matplotlib.rcParams["axes.unicode_minus"] = False
    if getattr(mtext.Text, "_bidi_patched", False):
        return
    _orig = mtext.Text.set_text
    def set_text(self, s):
        if isinstance(s, str) and getattr(self, "_bidi_display", None) == s:
            return _orig(self, s)
        fixed = fix_he(s)
        if isinstance(fixed, str):
            self._bidi_display = fixed
        return _orig(self, fixed)
    mtext.Text.set_text = set_text
    mtext.Text._bidi_patched = True

install_hebrew()

## 4. Stage registry and tolerant loaders

This is the machinery that lets the rest of the notebook run safely even on a partially run pipeline. `stage_dir(key)` locates a
stage by its two-digit prefix - so a folder renamed from `03_descriptive_analysis` to `03_network_descriptive_analysis` is still
found - and returns `None` when the stage was never run. `stage_file`, `load_csv` and `load_json` search a stage folder recursively
(stages put outputs either at the root or under `tables/`) and return `None` instead of raising, while printing what was found and
from where.

`rail_dir()` adds one deliberate fallback: notebook 11's rail tables also exist in the old read-only folder `outputs/rail/` from the
previous pipeline, so if stage 11 wasn't re-run, the rail section can still report numbers - and it prints which source it used.

Four small helpers complete the toolkit: `get()` pulls the first existing key from a dict or from a pandas `Series` (the schemas
drifted slightly between stages, for example `betweenness` versus `approx_betweenness`), `jget()` walks a path inside the nested
summary JSON files that stages 15-23 write (for example `jget(summary, 'graph', 'nodes')`) and returns `None` instead of raising
when some level is missing, `fmt()` formats a value for printing without raising on `None`/`NaN`, and `pct()` does the same for a
fraction in the range 0-1 shown as a percent.


In [ ]:
# --- Stage registry -------------------------------------------------------
STAGE_FOLDERS = {
    '00': '00_setup_and_data',
    '01': '01_data_preparation',
    '02': '02_graph_construction',
    '03': '03_descriptive_analysis',
    '04': '04_centrality_analysis',
    '05': '05_critical_station_isolation',
    '06': '06_robustness_analysis',
    '07': '07_regional_comparison',
    '09': '09_socioeconomic_equity',
    '08': '08_community_detection',
    '10': '10_network_model_comparison',
    '11': '11_rail_network_analysis',
    '12': '12_rail_socioeconomic',
    '13': '13_embeddings_link_prediction',
    '14': '14_multimodal_inventory',
    '15': '15_bus_network',
    '16': '16_lightrail_and_minor_modes',
    '17': '17_multimodal_transfer_hubs',
    '18': '18_travel_time_network',
    '19': '19_time_of_day_graphs',
    '20': '20_dynamic_resilience',
    '21': '21_demand_weighted_criticality',
    '22': '22_rerouting_model',
    '23': '23_critical_station_lenses',
}


def stage_dir(key):
    """Folder of stage `key` under outputs/nb, tolerating a renamed slug. None if absent."""
    if not OUT.is_dir():
        return None
    exact = OUT / STAGE_FOLDERS[key]
    if exact.is_dir():
        return exact
    for d in sorted(p for p in OUT.iterdir() if p.is_dir()):
        if d.name.startswith(key + '_'):
            return d
    return None


def stage_file(key, *patterns, root=None):
    """First file under a stage folder matching one of the glob patterns, else None."""
    base = root if root is not None else stage_dir(key)
    if base is None or not base.is_dir():
        return None
    for pat in patterns:
        direct = base / pat
        if direct.is_file():
            return direct
        hits = sorted(base.rglob(pat))
        if hits:
            return hits[0]
    return None


def load_csv(key, *patterns, root=None, quiet=False, **kwargs):
    """Read the first matching CSV of a stage; return None (with a message) if unavailable."""
    path = stage_file(key, *patterns, root=root)
    if path is None:
        if not quiet:
            print('  missing: ' + ' | '.join(patterns) + '  (stage ' + key + ')')
        return None
    try:
        df = pd.read_csv(path, encoding='utf-8-sig', **kwargs)
    except Exception as exc:
        print('  unreadable:', path, '->', exc)
        return None
    if not quiet:
        print('  loaded  :', path.relative_to(OUT) if OUT in path.parents else path,
              '(' + format(len(df), ',') + ' rows)')
    return df


def load_json(key, *patterns, root=None, quiet=False):
    """Read the first matching JSON of a stage; return None (with a message) if unavailable."""
    path = stage_file(key, *patterns, root=root)
    if path is None:
        if not quiet:
            print('  missing: ' + ' | '.join(patterns) + '  (stage ' + key + ')')
        return None
    try:
        with open(path, 'r', encoding='utf-8') as fh:
            payload = json.load(fh)
    except Exception as exc:
        print('  unreadable:', path, '->', exc)
        return None
    if not quiet:
        print('  loaded  :', path.relative_to(OUT) if OUT in path.parents else path)
    return payload


def rail_dir():
    """Stage 11 folder, or the legacy read-only outputs/rail folder as a fallback."""
    d = stage_dir('11')
    if d is not None:
        return d
    legacy = REPO / 'outputs' / 'rail'
    if legacy.is_dir():
        print('  note: stage 11 not found; falling back to the read-only legacy folder', legacy)
        return legacy
    return None


def get(container, *keys, default=None):
    """First present key of a dict / pandas Series, else `default`."""
    if container is None:
        return default
    for k in keys:
        if isinstance(container, dict):
            if k in container:
                return container[k]
        else:
            try:
                if k in container.index:
                    return container[k]
            except Exception:
                pass
    return default


def jget(payload, *path, default=None):
    """Walk a path through nested dicts (stage summary JSONs). Never raises."""
    cur = payload
    for key in path:
        if not isinstance(cur, dict) or key not in cur:
            return default
        cur = cur[key]
    return cur


def pick_col(df, *candidates):
    """First column name of `df` present in `candidates` (case-insensitive), else None."""
    if df is None:
        return None
    lower = {str(c).lower(): c for c in df.columns}
    for c in candidates:
        if c.lower() in lower:
            return lower[c.lower()]
    return None


def fmt(value, digits=3):
    """Print-safe number formatting: None / NaN / text never raise."""
    if value is None:
        return 'n/a'
    if isinstance(value, str):
        return value
    try:
        x = float(value)
    except (TypeError, ValueError):
        return str(value)
    if math.isnan(x):
        return 'n/a'
    if float(x).is_integer() and abs(x) >= 1:
        return format(int(x), ',')
    return format(x, ',.' + str(digits) + 'f')


def pct(value, digits=1):
    """Render a 0-1 share as a percentage string; never raises on None / NaN."""
    if value is None:
        return 'n/a'
    try:
        x = float(value)
    except (TypeError, ValueError):
        return 'n/a'
    if math.isnan(x):
        return 'n/a'
    return format(100.0 * x, ',.' + str(digits) + 'f') + '%'


print('registry ready:', len(STAGE_FOLDERS), 'upstream stages known.')


## 5. Which stages exist and which are missing

Before reading a single number we audit the pipeline. For each listed stage, the cell reports whether its folder exists, how many
tables and figures it contains, and when it was last modified. Missing stages are listed explicitly together with the notebook that
produces them, so a reviewer who ran only part of the pipeline sees immediately which sections below will be skipped, instead of
discovering it through a runtime error. The audit itself is saved to `tables/stage_availability.csv`.

Every section below is written to degrade to a printed message when its stage is missing, so a partial run produces a shorter report
rather than a stack trace.


In [ ]:
# --- Availability audit ---------------------------------------------------
import datetime

rows = []
for key, folder in STAGE_FOLDERS.items():
    d = stage_dir(key)
    if d is None:
        rows.append({'stage': key, 'expected_folder': folder, 'found_folder': '',
                     'present': False, 'n_tables': 0, 'n_figures': 0, 'last_modified': ''})
        continue
    tables = [p for p in d.rglob('*.csv')] + [p for p in d.rglob('*.json')]
    figures = [p for p in d.rglob('*.png')] + [p for p in d.rglob('*.svg')]
    files = [p for p in d.rglob('*') if p.is_file()]
    newest = max((p.stat().st_mtime for p in files), default=None)
    rows.append({
        'stage': key,
        'expected_folder': folder,
        'found_folder': d.name,
        # an empty folder means the stage started and failed - treat it as not run
        'present': len(files) > 0,
        'n_tables': len(tables),
        'n_figures': len(figures),
        'last_modified': ('' if newest is None else
                          datetime.datetime.fromtimestamp(newest).strftime('%Y-%m-%d %H:%M')),
    })

availability = pd.DataFrame(rows)
availability.to_csv(TABLES / 'stage_availability.csv', index=False, encoding='utf-8-sig')

present = availability[availability['present']]
absent = availability[~availability['present']]

print('stages present :', len(present), 'of', len(availability))
if len(absent):
    print()
    print('MISSING OR EMPTY - the matching sections below will be skipped:')
    for _, r in absent.iterrows():
        state = 'folder exists but is empty' if r['found_folder'] else 'no folder'
        print('   stage ' + r['stage'] + ' (' + state + ') -> run notebook '
              + r['expected_folder'])
else:
    print('all upstream stages found.')
print()
print('saved ->', TABLES / 'stage_availability.csv')
availability

## 6. The findings ledger

Every section below adds its headline numbers to one list via `record(...)`, and section 25 turns that list into the cross-stage
summary table. Doing it this way guarantees the summary table never drifts from the sections: there is exactly one place where a
number enters the notebook - the file it's read from - and the table is built from those same values. `record` also keeps the
source filename, so every row in the final table can be traced back to disk.


In [ ]:
# --- The single ledger every section writes into --------------------------
FINDINGS = []


def record(stage, theme, metric, value, source, note=''):
    """Append one traceable headline number to the cross-stage summary table."""
    FINDINGS.append({
        'stage': stage,
        'theme': theme,
        'metric': metric,
        'value': value if isinstance(value, str) else fmt(value),
        'source_file': source,
        'note': note,
    })


def section_header(title):
    print('=' * 78)
    print(title)
    print('=' * 78)


print('ledger initialised.')

## 7. The scale and shape of the network (stages 00-03)

The baseline every other finding is measured against: how many stations and segments the travel-adjacency graph has, how sparse it
is, how fragmented it already is before anyone attacks it, and how many single structural failure points (articulation points,
bridges) it contains. Stage 02 records the build statistics, stage 03 the descriptive measurements; either can carry the numbers,
so the cell reads both and prefers stage 03's `network_summary.json` where the two overlap. Nothing is recomputed - the graph isn't
even loaded here.


In [ ]:
# --- Scale of the network -------------------------------------------------
section_header('SCALE OF THE NETWORK (stages 00-03)')

feed = load_json('00', 'feed_manifest.json', quiet=True)
build = load_json('02', 'graph_build_summary.json')
desc = load_json('03', 'network_summary.json')
if desc is None:
    desc_csv = load_csv('03', 'network_summary.csv')
    desc = desc_csv.iloc[0] if desc_csv is not None and len(desc_csv) else None

ap_df = load_csv('03', 'articulation_points.csv', 'top_articulation_points.csv')
br_df = load_csv('03', 'bridges.csv', 'top_bridges.csv')

if desc is None and build is None:
    print('\nSKIPPED - neither stage 02 nor stage 03 has been run; no network summary to report.')
    N_NODES = None
else:
    N_NODES = get(desc, 'num_nodes', default=get(build, 'num_nodes'))
    N_EDGES = get(desc, 'num_edges_undirected', default=get(build, 'num_edges_undirected'))
    DENSITY = get(desc, 'density', default=get(build, 'density'))
    AVG_DEG = get(desc, 'avg_degree', default=get(build, 'avg_degree'))
    N_COMP = get(desc, 'num_connected_components', default=get(build, 'connected_components'))
    LCC_SHARE = get(desc, 'largest_component_share')
    if LCC_SHARE is None:
        lcc_nodes = get(desc, 'largest_component_nodes', default=get(build, 'largest_component_size'))
        LCC_SHARE = (float(lcc_nodes) / float(N_NODES)) if (lcc_nodes and N_NODES) else None
    N_AP = get(desc, 'num_articulation_points', default=(len(ap_df) if ap_df is not None else None))
    N_BR = get(desc, 'num_bridges', default=(len(br_df) if br_df is not None else None))

    print()
    print('stations (nodes)            :', fmt(N_NODES))
    print('segments (undirected edges) :', fmt(N_EDGES))
    print('density                     :', fmt(DENSITY, 6))
    print('average degree              :', fmt(AVG_DEG, 2))
    print('connected components        :', fmt(N_COMP))
    print('largest-component share     :',
          'n/a' if LCC_SHARE is None else format(100 * float(LCC_SHARE), '.2f') + '%')
    print('articulation points         :', fmt(N_AP),
          '' if (N_AP is None or not N_NODES) else
          '(' + format(100 * float(N_AP) / float(N_NODES), '.2f') + '% of stations)')
    print('bridges                     :', fmt(N_BR),
          '' if (N_BR is None or not N_EDGES) else
          '(' + format(100 * float(N_BR) / float(N_EDGES), '.2f') + '% of segments)')

    src = 'network_summary.json (03) / graph_build_summary.json (02)'
    record('02-03', 'scale', 'active stations', N_NODES, src)
    record('02-03', 'scale', 'undirected segments', N_EDGES, src)
    record('02-03', 'scale', 'density', fmt(DENSITY, 6), src)
    record('02-03', 'scale', 'average degree', fmt(AVG_DEG, 2), src)
    record('02-03', 'scale', 'largest-component share',
           'n/a' if LCC_SHARE is None else format(100 * float(LCC_SHARE), '.2f') + '%', src)
    record('03', 'structure', 'articulation points (cut vertices)', N_AP,
           'articulation_points.csv', 'stations whose removal disconnects the graph')
    record('03', 'structure', 'bridges (cut edges)', N_BR, 'bridges.csv',
           'segments whose removal disconnects the graph')

if feed is not None:
    print()
    print('feed manifest keys available from stage 00:', ', '.join(sorted(feed)[:12]))

## 8. Finding 1 - operational criticality is not structural criticality

This is the project's central result, so it gets the most careful treatment.

* **Operational criticality** = how much service a station carries. In the travel-adjacency graph this is **weighted degree**: the
  number of scheduled trip records touching the station's segments. It's what an operator sees on their dashboard.
* **Structural criticality** = what the network loses topologically if the station disappears. The two proxies for it here are
  **betweenness centrality** (how many shortest paths pass through the station) and **articulation-point status** (whether removing
  it actually disconnects the graph).

The cell measures the gap in three ways, all read from stage 04's `stop_metrics.csv`, and where present, also from stage 04's saved
Spearman matrix:

1. **Rank correlation** between weighted degree and betweenness across all stations. Spearman rather than Pearson because both
   distributions are heavy-tailed.
2. **Top-K overlap** at several ranking depths. This is the operationally meaningful quantity: if a planner protects the K busiest
   stations, how many of the K most structurally critical did they cover? Overlap well below 1.0 means the two lists are mostly
   made of different stations.
3. **Articulation points among the busiest stations** - the sharpest version of the same question, since a cut node is a *proven*
   single failure point and not a centrality estimate.

The overlap table is saved so the figure can be redrawn without rerunning the notebook.


In [ ]:
# --- Operational vs structural criticality --------------------------------
section_header('FINDING 1 - OPERATIONAL vs STRUCTURAL CRITICALITY (stage 04)')

metrics = load_csv('04', 'stop_metrics.csv', dtype={'stop_id': str})
corr_matrix = load_csv('04', 'centrality_correlation_spearman.csv', index_col=0)
stability = load_csv('04', 'betweenness_stability.csv', quiet=True)

WD_COL = BT_COL = None
overlap_table = None
rho_wd_bt = None

if metrics is None:
    print('\nSKIPPED - stage 04 (centrality analysis) has not been run, so the central '
          'operational-vs-structural comparison cannot be made.')
else:
    WD_COL = pick_col(metrics, 'weighted_degree', 'weighted_degree_centrality')
    BT_COL = pick_col(metrics, 'approx_betweenness', 'betweenness', 'betweenness_centrality')
    DEG_COL = pick_col(metrics, 'degree')
    ID_COL = pick_col(metrics, 'stop_id')
    NAME_COL = pick_col(metrics, 'stop_name', 'label')

    if WD_COL is None or BT_COL is None or ID_COL is None:
        print('\nSKIPPED - stop_metrics.csv lacks a stop_id, weighted-degree or betweenness '
              'column. Found:', list(metrics.columns))
        WD_COL = BT_COL = None
    else:
        for c in (WD_COL, BT_COL) + ((DEG_COL,) if DEG_COL else ()):
            metrics[c] = pd.to_numeric(metrics[c], errors='coerce')
        work = metrics.dropna(subset=[WD_COL, BT_COL]).copy()

        # (1) rank correlation - prefer the matrix stage 04 already computed
        if corr_matrix is not None and WD_COL in corr_matrix.columns and BT_COL in corr_matrix.index:
            rho_wd_bt = float(corr_matrix.loc[BT_COL, WD_COL])
            rho_source = 'centrality_correlation_spearman.csv'
        else:
            rho_wd_bt = float(work[WD_COL].corr(work[BT_COL], method='spearman'))
            rho_source = 'stop_metrics.csv (recomputed here)'

        print()
        print('Spearman rho, weighted degree vs betweenness :',
              format(rho_wd_bt, '+.3f'), ' <-', rho_source)
        if corr_matrix is not None and DEG_COL and DEG_COL in corr_matrix.columns \
                and BT_COL in corr_matrix.index:
            print('Spearman rho, plain degree     vs betweenness :',
                  format(float(corr_matrix.loc[BT_COL, DEG_COL]), '+.3f'))

        # (2) top-K overlap between the two rankings
        rows = []
        for k in OVERLAP_KS:
            k_eff = min(k, len(work))
            top_wd = set(work.nlargest(k_eff, WD_COL)[ID_COL].astype(str))
            top_bt = set(work.nlargest(k_eff, BT_COL)[ID_COL].astype(str))
            shared = len(top_wd & top_bt)
            rows.append({'k': k_eff, 'shared_stations': shared,
                         'overlap_share': round(shared / k_eff, 4) if k_eff else np.nan})
        overlap_table = pd.DataFrame(rows)
        overlap_table.to_csv(TABLES / 'operational_vs_structural_overlap.csv',
                             index=False, encoding='utf-8-sig')
        print()
        print('Top-K overlap between the service ranking and the betweenness ranking:')
        print(overlap_table.to_string(index=False))
        print('saved ->', TABLES / 'operational_vs_structural_overlap.csv')

        # (3) how many of the busiest stations are proven cut vertices
        ap_overlap = None
        if ap_df is not None:
            ap_id = pick_col(ap_df, 'stop_id')
            if ap_id is not None:
                ap_ids = set(ap_df[ap_id].astype(str))
                k_eff = min(100, len(work))
                busiest = set(work.nlargest(k_eff, WD_COL)[ID_COL].astype(str))
                most_central = set(work.nlargest(k_eff, BT_COL)[ID_COL].astype(str))
                ap_overlap = (len(busiest & ap_ids), len(most_central & ap_ids), k_eff)
                print()
                print('Of the ' + str(k_eff) + ' busiest stations (weighted degree), '
                      + str(ap_overlap[0]) + ' are articulation points.')
                print('Of the ' + str(k_eff) + ' highest-betweenness stations,      '
                      + str(ap_overlap[1]) + ' are articulation points.')

        shallow = overlap_table[overlap_table['k'] <= 50]
        k50 = (shallow if len(shallow) else overlap_table).iloc[-1]
        record('04', 'operational vs structural', 'Spearman rho (weighted degree vs betweenness)',
               format(rho_wd_bt, '+.3f'), rho_source,
               'far below 1: the two rankings are not the same stations')
        record('04', 'operational vs structural',
               'top-' + str(int(k50['k'])) + ' overlap between the rankings',
               format(100 * float(k50['overlap_share']), '.1f') + '%',
               'stop_metrics.csv', 'share of the busiest stations that are also the most central')
        if ap_overlap is not None:
            record('04+03', 'operational vs structural',
                   'articulation points among the ' + str(ap_overlap[2]) + ' busiest stations',
                   ap_overlap[0], 'stop_metrics.csv + articulation_points.csv')

        # illustrative head-to-head table
        cols = [c for c in [ID_COL, NAME_COL, DEG_COL, WD_COL, BT_COL] if c]
        print()
        print('Top ' + str(TOP_N) + ' by service volume (operational):')
        display(work.nlargest(TOP_N, WD_COL)[cols].reset_index(drop=True))
        print('Top ' + str(TOP_N) + ' by betweenness (structural):')
        display(work.nlargest(TOP_N, BT_COL)[cols].reset_index(drop=True))

if stability is not None and len(stability):
    print()
    print('Sanity check on the betweenness estimate itself (stage 04, two random seeds):')
    print(stability.iloc[0].to_string())

### Figure 1 - the gap, drawn

Two panels, both computed from the same `stop_metrics.csv`:

* **Left** - every station as one point: service volume on the x-axis (log scale, because weighted degree spans several orders of
  magnitude) against betweenness on the y-axis (symmetric-log scale, because a large share of stations get an estimated betweenness
  of exactly zero - they were never on a sampled shortest path). If the two notions of criticality agreed, the point cloud would
  collapse onto a rising line. The Spearman rho printed in the corner is the number from the cell above, not a recomputation.
* **Right** - top-K agreement between the two rankings at each depth in `OVERLAP_KS`, with a dashed line at 1.0 marking perfect
  agreement. This is the panel to cite to a planner.

If stage 04 is missing, the figure is skipped with a message.


In [ ]:
# --- Figure 1: operational vs structural criticality -----------------------
if metrics is None or WD_COL is None or BT_COL is None or overlap_table is None:
    print('SKIPPED - Figure 1 needs stage 04 stop_metrics.csv with weighted-degree '
          'and betweenness columns.')
else:
    d = metrics[[WD_COL, BT_COL]].apply(pd.to_numeric, errors='coerce').dropna()
    d = d[d[WD_COL] > 0]

    fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.8))

    ax = axes[0]
    ax.scatter(d[WD_COL], d[BT_COL], s=7, alpha=0.22, color='#2563eb', linewidths=0)
    ax.set_xscale('log')
    positive = d[d[BT_COL] > 0][BT_COL]
    linthresh = float(positive.min()) if len(positive) else 1e-9
    ax.set_yscale('symlog', linthresh=max(linthresh, 1e-12))
    ax.set_xlabel('Weighted degree - scheduled trip records (log scale)')
    ax.set_ylabel('Betweenness centrality (symlog)')
    ax.set_title('Operational load vs structural centrality')
    ax.text(0.03, 0.95,
            'Spearman rho = ' + format(rho_wd_bt, '+.3f') + '\n'
            + format(100 * float((d[BT_COL] <= 0).mean()), '.1f')
            + '% of stations have zero estimated betweenness',
            transform=ax.transAxes, va='top', ha='left', fontsize=10,
            bbox=dict(boxstyle='round,pad=0.4', fc='#f8fafc', ec='#cbd5e1'))

    ax = axes[1]
    labels = ['top ' + format(int(k), ',') for k in overlap_table['k']]
    bars = ax.bar(labels, overlap_table['overlap_share'], color='#7c3aed', edgecolor='white')
    for bar, share, shared in zip(bars, overlap_table['overlap_share'],
                                  overlap_table['shared_stations']):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
                format(100 * float(share), '.0f') + '%\n(' + format(int(shared), ',') + ')',
                ha='center', va='bottom', fontsize=10)
    ax.axhline(1.0, color='#334155', ls='--', lw=1.0, label='perfect agreement')
    ax.set_ylim(0, 1.18)
    ax.set_ylabel('share of the two top-K lists that coincide')
    ax.set_title('How much do the busiest and the most central stations overlap?')
    ax.legend(loc='upper right', fontsize=9)

    fig.suptitle('Finding 1: protecting the busiest stations is not protecting the network',
                 fontsize=13)
    fig.tight_layout()
    fig.savefig(FIGURES / 'operational_vs_structural.png', dpi=FIG_DPI, bbox_inches='tight')
    plt.show()
    print('saved ->', FIGURES / 'operational_vs_structural.png')

## 9. Finding 2 - how many critical stations have no substitute (stage 05)

An articulation point or a high-betweenness station is a true single failure point only if a stranded passenger has nowhere to go.
Stage 05 answers this with a purely spatial test: for each station in the top 10% of betweenness, how far is the nearest *other*
station? Anything within a short walk counts as effectively substitutable, and anything beyond that counts as isolated.

This cell reports the resulting distance bands and the isolation share. The test's weakness is noted in the limitations section and
is worth keeping in mind when reading the number: a neighboring station 300 m away is a substitute only if it's served by a line
that reaches some useful destination, and the test doesn't check that at all.


In [ ]:
# --- Substitutability of critical stations --------------------------------
section_header('FINDING 2 - DO CRITICAL STATIONS HAVE A SUBSTITUTE? (stage 05)')

iso_summary = load_csv('05', 'isolation_summary.csv')
iso_detail = load_csv('05', 'critical_isolation.csv', dtype={'stop_id': str}, quiet=True)

if iso_summary is None:
    print('\nSKIPPED - stage 05 (critical-station isolation) has not been run.')
else:
    band_col = pick_col(iso_summary, 'distance_band')
    n_col = pick_col(iso_summary, 'n_critical_stops', 'n_stops', 'count')
    verdict_col = pick_col(iso_summary, 'verdict')
    total_critical = int(pd.to_numeric(iso_summary[n_col], errors='coerce').fillna(0).sum())

    print()
    print(iso_summary.to_string(index=False))
    print()
    print('critical stations examined :', format(total_critical, ','))

    if verdict_col is not None and total_critical:
        isolated_n = int(pd.to_numeric(
            iso_summary.loc[iso_summary[verdict_col].astype(str).str.contains('isolat', case=False),
                            n_col], errors='coerce').fillna(0).sum())
        print('genuinely isolated         :', format(isolated_n, ','),
              '(' + format(100 * isolated_n / total_critical, '.1f') + '% of critical stations)')
        record('05', 'substitutability', 'critical stations examined (top 10% betweenness)',
               total_critical, 'isolation_summary.csv')
        record('05', 'substitutability', 'critical stations with no nearby alternative',
               format(100 * isolated_n / total_critical, '.1f') + '%', 'isolation_summary.csv',
               'spatial proximity only - service substitutability is not tested')

    if iso_detail is not None:
        bt = pick_col(iso_detail, 'betweenness', 'approx_betweenness')
        dist = pick_col(iso_detail, 'nearest_alt_m')
        name = pick_col(iso_detail, 'stop_name')
        flag = pick_col(iso_detail, 'is_isolated')
        cols = [c for c in [pick_col(iso_detail, 'stop_id'), name, bt, dist] if c]
        if flag is not None:
            worst = iso_detail[iso_detail[flag].astype(str).str.lower().isin(['true', '1'])]
        else:
            worst = iso_detail
        if len(worst) and bt:
            print()
            print('Highest-betweenness isolated stations:')
            display(worst.sort_values(bt, ascending=False)[cols].head(TOP_N).reset_index(drop=True))

## 10. Finding 3 - targeted attack versus random failure (stage 06)

Stage 06 removes stations from the graph by one strategy at a time and records the size of the largest remaining connected
component. Five targeted removal orders (degree, weighted degree, PageRank, betweenness, articulation points) are compared against a
random baseline with a fixed seed, averaged over several trials.

The cell reads three outputs and reports on them without re-running the simulation:

* `auc_summary.csv` - the area under each resilience curve, that is, the average share of the largest component over the whole
  attack. Lower means a more damaging strategy. It's reported under **both** normalizations stage 06 exports, and this matters:
  `lcc_share_original` divides by the original station count, so it drops even when the network didn't fragment at all (the removed
  stations are simply gone), whereas `lcc_share_surviving` divides by the stations that still exist and therefore measures actual
  fragmentation. The honest headline uses the surviving-normalized figure.
* `damage_snapshots.csv` - the state after a small number of removals, which is the operationally relevant regime.
* `disruption_results.csv` - the full curves, used for the summary figure later.


In [ ]:
# --- Resilience under attack ----------------------------------------------
section_header('FINDING 3 - TARGETED ATTACK vs RANDOM FAILURE (stage 06)')

auc_df = load_csv('06', 'auc_summary.csv')
snapshots = load_csv('06', 'damage_snapshots.csv')
disruption = load_csv('06', 'disruption_results.csv')

if auc_df is None and disruption is None:
    print('\nSKIPPED - stage 06 (robustness analysis) has not been run.')
else:
    if auc_df is not None:
        print()
        print('Area under the resilience curve (lower = the attack does more damage):')
        print(auc_df.to_string(index=False))

        strat_col = pick_col(auc_df, 'strategy')
        surv_col = pick_col(auc_df, 'auc_surviving')
        orig_col = pick_col(auc_df, 'auc_original')
        if strat_col and surv_col:
            random_mask = auc_df[strat_col].astype(str).str.contains('random', case=False)
            targeted = auc_df[~random_mask]
            if len(targeted):
                worst = targeted.loc[pd.to_numeric(targeted[surv_col],
                                                   errors='coerce').idxmin()]
                print()
                print('most damaging targeted strategy :', worst[strat_col],
                      '(AUC surviving-normalised =', fmt(worst[surv_col], 4) + ')')
                record('06', 'resilience', 'most damaging attack strategy',
                       str(worst[strat_col]), 'auc_summary.csv',
                       'ranked by area under the surviving-normalised resilience curve')
                record('06', 'resilience', 'AUC of the most damaging targeted attack',
                       fmt(worst[surv_col], 4), 'auc_summary.csv',
                       'mean largest-component share among surviving stations')
            if random_mask.any():
                rnd = auc_df[random_mask].iloc[0]
                print('random-failure baseline         : AUC surviving-normalised =',
                      fmt(rnd[surv_col], 4))
                record('06', 'resilience', 'AUC of the random-failure baseline',
                       fmt(rnd[surv_col], 4), 'auc_summary.csv',
                       'the gap to the targeted attack is the robust-yet-fragile signature')
                if len(targeted):
                    gap = float(rnd[surv_col]) - float(worst[surv_col])
                    print('gap (random - worst targeted)   :', format(gap, '.4f'))
                    record('06', 'resilience', 'targeted-vs-random AUC gap',
                           format(gap, '.4f'), 'auc_summary.csv',
                           'positive = targeted removal hurts more than random failure')
            if orig_col:
                print()
                print('NOTE: the original-normalised AUC column is kept only for comparability with '
                      'the earlier report. It divides by the original station count and therefore '
                      'falls even when nothing fragments; do not quote it as fragmentation.')

    if snapshots is not None and len(snapshots):
        rem_col = pick_col(snapshots, 'removed')
        if rem_col:
            smallest_k = int(pd.to_numeric(snapshots[rem_col], errors='coerce').min())
            view = snapshots[pd.to_numeric(snapshots[rem_col], errors='coerce') == smallest_k]
            print()
            print('State after removing only ' + format(smallest_k, ',') + ' stations:')
            print(view.to_string(index=False))
            surv = pick_col(view, 'lcc_share_surviving')
            strat = pick_col(view, 'strategy')
            if surv and strat and len(view):
                v = view.loc[pd.to_numeric(view[surv], errors='coerce').idxmin()]
                record('06', 'resilience',
                       'worst largest-component share after removing '
                       + format(smallest_k, ',') + ' stations',
                       fmt(v[surv], 4), 'damage_snapshots.csv',
                       'strategy: ' + str(v[strat]) + '; normalised by surviving stations')

## 11. Finding 4 - the regional differences are statistically significant but substantively small (stage 07)

Stage 07 splits the stations by administrative district and by metropolitan area and asks whether criticality is distributed
unevenly. It reports both a chi-square test of independence and Cramer's V - and that pairing is the whole point. With tens of
thousands of stations, a chi-square test will find *any* difference significant, so the p-value alone is almost meaningless here;
the effect size is what tells you whether the difference matters. The cell prints both, along with the percentage-point gap between
the highest and lowest region, and flags small-sample groups that stage 07 marked.


In [ ]:
# --- Regional comparison --------------------------------------------------
section_header('FINDING 4 - REGIONAL DIFFERENCES (stage 07)')

regional = load_csv('07', 'regional_summary.csv')
regional_sig = load_csv('07', 'regional_significance.csv')
metro = load_csv('07', 'metro_summary.csv', quiet=True)

if regional is None:
    print('\nSKIPPED - stage 07 (regional comparison) has not been run.')
else:
    print()
    print(regional.to_string(index=False))

    reg_col = pick_col(regional, 'region', 'group')
    pct_col = pick_col(regional, 'pct_critical')
    small_col = pick_col(regional, 'small_sample')
    if pct_col and reg_col and pd.to_numeric(regional[pct_col], errors='coerce').notna().any():
        pct_series = pd.to_numeric(regional[pct_col], errors='coerce')
        hi = regional.loc[pct_series.idxmax()]
        lo = regional.loc[pct_series.idxmin()]
        print()
        print('highest share of critical stops :', str(hi[reg_col]),
              format(float(hi[pct_col]), '.2f') + '%')
        print('lowest  share of critical stops :', str(lo[reg_col]),
              format(float(lo[pct_col]), '.2f') + '%')
        record('07', 'regional', 'region with the highest share of critical stops',
               str(hi[reg_col]) + ' (' + format(float(hi[pct_col]), '.2f') + '%)',
               'regional_summary.csv')
        record('07', 'regional', 'spread between the extreme regions',
               format(float(pct_series.max() - pct_series.min()), '.2f') + ' pp', 'regional_summary.csv')
    if small_col is not None and reg_col and pct_col:
        flagged = regional[regional[small_col].astype(str).str.lower().isin(['true', '1'])]
        if len(flagged):
            print()
            print('small-sample groups flagged by stage 07 - do not quote their percentages:')
            print(flagged[[reg_col, pct_col]].to_string(index=False))

    if regional_sig is not None and len(regional_sig):
        row = regional_sig.iloc[0]
        p_val = get(row, 'p_value')
        v = get(row, 'cramers_v')
        print()
        print('chi-square test of independence (region x is_critical):')
        print(row.to_string())
        if v is not None:
            strength = ('negligible' if float(v) < 0.1 else 'small' if float(v) < 0.3
                        else 'moderate' if float(v) < 0.5 else 'large')
            print()
            print("Cramer's V =", fmt(v, 4), '->', strength, 'association.')
            print('p =', fmt(p_val, 6),
                  '- at this sample size significance was almost guaranteed; the effect size, '
                  'not the p-value, is what should be quoted.')
            record('07', 'regional', "Cramer's V (region x criticality)",
                   fmt(v, 4) + ' (' + strength + ')', 'regional_significance.csv',
                   'significant at p = ' + fmt(p_val, 6) + ' but substantively ' + strength)

    if metro is not None:
        print()
        print('Metropolitan areas:')
        print(metro.to_string(index=False))

## 12. Finding 5 - the socioeconomic equity result is a Simpson's paradox (stage 09)

Stage 09 links each station to a Central Bureau of Statistics statistical area and asks whether scheduled service per capita
correlates with the area's socioeconomic cluster. It computes the correlation at the national level, at the community level, and
separately *within* individual cities, with Benjamini-Hochberg and Bonferroni corrections across the family of within-city tests.

The headline isn't "service is inequitable" - it's that the sign of the relationship depends on the level of aggregation. A national
correlation pointing one way while the within-city correlations point the opposite way is a classic Simpson's paradox, and it means
any single-number claim about transport equity in Israel derived from this data is a claim about the chosen aggregation level, not
about the country. The cell reads `socioeconomic_summary.json` (which stores the sign convention explicitly) and prints the
multiple-comparisons bookkeeping alongside the correlations.


In [ ]:
# --- Socio-economic equity ------------------------------------------------
section_header('FINDING 5 - SOCIO-ECONOMIC EQUITY (stage 09)')

socio = load_json('09', 'socioeconomic_summary.json')
within = load_csv('09', 'socioeconomic_within_cluster_correlation.csv', quiet=True)
national_corr = load_csv('09', 'socioeconomic_national_correlations.csv', quiet=True)

if socio is None:
    print('\nSKIPPED - stage 09 (socio-economic equity) has not been run.')
else:
    print()
    print('sign convention  :', get(socio, 'sign_convention', default='n/a'))
    print('service variable :', get(socio, 'service_variable', default='n/a'))
    print()

    nat_rho = get(socio, 'national_neighborhood_rho')
    nat_p = get(socio, 'national_neighborhood_p')
    nat_n = get(socio, 'national_neighborhood_n')
    com_rho = get(socio, 'community_level_rho')
    com_p = get(socio, 'community_level_p')
    com_n = get(socio, 'community_level_n')
    rho_min = get(socio, 'within_rho_min')
    rho_max = get(socio, 'within_rho_max')
    n_tests = get(socio, 'communities_tested')
    sig_raw = get(socio, 'significant_raw')
    sig_fdr = get(socio, 'significant_fdr_bh')
    sig_bonf = get(socio, 'significant_bonferroni')

    print('national, neighbourhood level : rho =', fmt(nat_rho, 3),
          ' p =', fmt(nat_p, 6), ' n =', fmt(nat_n))
    print('community level (aggregated)  : rho =', fmt(com_rho, 3),
          ' p =', fmt(com_p, 6), ' n =', fmt(com_n))
    print('within individual communities : rho ranges from', fmt(rho_min, 3),
          'to', fmt(rho_max, 3), 'over', fmt(n_tests), 'communities')
    print('significant within-community tests: raw', fmt(sig_raw),
          '| BH-FDR', fmt(sig_fdr), '| Bonferroni', fmt(sig_bonf))
    print('expected false positives at the raw alpha:',
          fmt(get(socio, 'expected_false_positives_at_raw_alpha'), 2))

    if nat_rho is not None and rho_min is not None and rho_max is not None:
        try:
            flips = (float(nat_rho) < 0 <= float(rho_max)) or (float(nat_rho) > 0 >= float(rho_min))
        except (TypeError, ValueError):
            flips = False
        print()
        if flips:
            print("SIMPSON'S PARADOX: the national correlation and the within-city correlations "
                  'do not share a sign. The aggregation level decides the answer, so no single '
                  'number describes "equity" here.')
        else:
            print('The national and within-city correlations share a sign; read the ranges above '
                  'before quoting any single figure.')

    record('09', 'equity', 'national service-per-capita vs socio-economic cluster (Spearman rho)',
           fmt(nat_rho, 3), 'socioeconomic_summary.json',
           'p = ' + fmt(nat_p, 6) + ', n = ' + fmt(nat_n))
    record('09', 'equity', 'within-community rho range',
           fmt(rho_min, 3) + ' to ' + fmt(rho_max, 3), 'socioeconomic_summary.json',
           'sign differs from the national figure -> Simpson\'s paradox')
    record('09', 'equity', 'within-community tests significant after BH-FDR',
           fmt(sig_fdr) + ' of ' + fmt(n_tests),
           'socioeconomic_within_cluster_correlation.csv')

    if within is not None:
        print()
        print('Per-community correlations:')
        display(within.head(TOP_N))
    if national_corr is not None:
        print('National correlations by metric:')
        display(national_corr.head(TOP_N))

## 13. Community structure (stage 08)

Stage 08 partitions the network with Louvain and reports the modularity, the distribution of community sizes, the share of edges
that cross a community boundary, and - importantly - a seed and resolution sweep showing how much the *number* of communities moves
between runs while the partition itself stays largely the same. That sweep is why this section cites a **range** of community counts
rather than a single number: the count is the least stable thing Louvain produces, and two runs of the same algorithm on the same
graph disagree about it legitimately.

The cell reads stage 08's headline JSON and its community table, and falls back to the community table stage 09 writes as a
by-product if stage 08 wasn't run. If nothing is found, it prints a skip message and continues - no section below depends on it.


In [ ]:
# --- Community structure --------------------------------------------------
section_header('COMMUNITY STRUCTURE (stage 08)')

comm_json = load_json('08', 'community_detection_summary.json', 'community_summary.json')
comm = load_csv('08', 'community_summary.csv', 'communities.csv')
comm_assign = load_csv('08', 'community_assignments.csv', quiet=True)
comm_socio = load_csv('09', 'community_socioeconomic_summary.csv', quiet=True)

if comm_json is None and comm is None and comm_assign is None and comm_socio is None:
    print('\nSKIPPED - no community-detection artifacts found under stage 08 '
          '(' + STAGE_FOLDERS['08'] + ') or stage 09. Run notebook 08 to populate this section.')
else:
    if comm_json is not None:
        n_comm = get(comm_json, 'num_communities')
        modularity = get(comm_json, 'modularity')
        cross_share = get(comm_json, 'cross_community_edge_share')
        lo = get(comm_json, 'seed_sweep_min_communities')
        hi = get(comm_json, 'seed_sweep_max_communities')
        ari_min = get(comm_json, 'seed_sweep_min_ari_vs_headline')
        print()
        print('communities (headline run)   :', fmt(n_comm))
        print('modularity                   :', fmt(modularity, 4))
        print('cross-community edge share   :',
              'n/a' if cross_share is None else format(100 * float(cross_share), '.2f') + '%')
        if lo is not None and hi is not None:
            print('community count across seeds :', fmt(lo), 'to', fmt(hi),
                  '- the count is the least stable output of Louvain')
        if ari_min is not None:
            print('worst seed-to-seed ARI       :', fmt(ari_min, 3),
                  '- the partition itself is far more stable than its cardinality')

        record('08', 'communities', 'communities detected (headline run)', n_comm,
               'community_detection_summary.json')
        record('08', 'communities', 'modularity', fmt(modularity, 4),
               'community_detection_summary.json',
               'high modularity = the network really does split into regional clusters')
        if lo is not None and hi is not None:
            record('08', 'communities', 'community count across random seeds',
                   fmt(lo) + ' to ' + fmt(hi), 'louvain_stability.csv',
                   'quote the range, not a single count')
        if cross_share is not None:
            record('08', 'communities', 'share of edges crossing a community boundary',
                   format(100 * float(cross_share), '.2f') + '%',
                   'community_detection_summary.json')

    if comm is not None:
        print()
        print('Largest communities:')
        display(comm.head(TOP_N))
    if comm_assign is not None:
        cid = pick_col(comm_assign, 'community_id', 'louvain_community', 'community')
        if cid is not None:
            print('communities in the assignment table:',
                  format(int(comm_assign[cid].nunique()), ','))
    if comm_socio is not None:
        print()
        print('Community-level socio-economic summary (from stage 09):')
        display(comm_socio.head(TOP_N))

## 14. What kind of network is this? (stage 10)

Stage 10 rebuilds four null models fitted to the real graph - Erdos-Renyi, a configuration model on the observed degree sequence,
Barabasi-Albert and Watts-Strogatz - and compares degree distribution (Kolmogorov-Smirnov distance), clustering coefficient, path
length and small-world sigma. This tells you which generative story the real network is closest to, and therefore which theoretical
expectations about resilience can be carried over to it. The cell reads the three comparison tables and reports the best-matching
model by KS distance, along with the real network's small-world ratios.


In [ ]:
# --- Null-model comparison ------------------------------------------------
section_header('NETWORK MODEL COMPARISON (stage 10)')

model_cmp = load_csv('10', 'network_model_comparison.csv')
deg_summary = load_csv('10', 'degree_distribution_summary.csv')
small_world = load_csv('10', 'small_world_ratios.csv')

if model_cmp is None and deg_summary is None:
    print('\nSKIPPED - stage 10 (network model comparison) has not been run.')
else:
    if model_cmp is not None:
        print()
        display(model_cmp.round(4))
    if deg_summary is not None:
        mcol = pick_col(deg_summary, 'model')
        kcol = pick_col(deg_summary, 'ks_vs_real')
        if mcol and kcol:
            others = deg_summary[~deg_summary[mcol].astype(str).str.contains('real', case=False)]
            if len(others):
                best = others.loc[pd.to_numeric(others[kcol], errors='coerce').idxmin()]
                print()
                print('closest degree distribution to the real network:', best[mcol],
                      '(KS =', fmt(best[kcol], 4) + ')')
                record('10', 'network model', 'null model closest to the real degree distribution',
                       str(best[mcol]) + ' (KS = ' + fmt(best[kcol], 4) + ')',
                       'degree_distribution_summary.csv')
        acol = pick_col(deg_summary, 'hill_alpha_kmin4')
        if mcol and acol:
            real = deg_summary[deg_summary[mcol].astype(str).str.contains('real', case=False)]
            if len(real):
                record('10', 'network model', 'Hill tail exponent of the real degree distribution',
                       fmt(real.iloc[0][acol], 3), 'degree_distribution_summary.csv',
                       'descriptive tail estimate, not a validated power-law fit')
    if small_world is not None:
        print()
        print('Small-world ratios:')
        print(small_world.round(3).to_string(index=False))
        mcol = pick_col(small_world, 'model')
        scol = pick_col(small_world, 'small_world_sigma')
        if mcol and scol:
            real = small_world[small_world[mcol].astype(str).str.contains('real', case=False)]
            if len(real):
                record('10', 'network model', 'small-world sigma of the real network',
                       fmt(real.iloc[0][scol], 3), 'small_world_ratios.csv',
                       'sigma >> 1 indicates small-world structure')

## 15. Finding 6 - the rail sub-network is a different regime (stage 11)

Filtering the file to `route_type = 2` in the GTFS (heavy rail, Israel Railways) leaves a network two orders of magnitude smaller
than the bus network - a few dozen stations arranged in a near-path-graph structure with a handful of branches. Because it's almost
a tree, almost every interior station is a cut node, so the structural fragility that is a minority phenomenon in the bus network is
the *normal* case here. Stage 11 also runs an exhaustive single-station removal over every station (feasible at this scale, unlike
the bus network), and this is the strongest form of the criticality question the project asks anywhere.

The cell reads stage 11's summary and damage table; if stage 11 wasn't re-run, it falls back to the old read-only folder
`outputs/rail/` and notes this.


In [ ]:
# --- Rail sub-network -----------------------------------------------------
section_header('FINDING 6 - THE RAIL SUB-NETWORK (stage 11)')

RAIL = rail_dir()
rail_summary = load_json('11', 'summary.json', root=RAIL)
rail_damage = load_csv('11', 'single_station_damage.csv', root=RAIL, dtype={'stop_id': str})
rail_metrics = load_csv('11', 'rail_station_metrics.csv', root=RAIL,
                        dtype={'stop_id': str}, quiet=True)

if rail_summary is None and rail_damage is None:
    print('\nSKIPPED - stage 11 (rail network analysis) has not been run and no legacy '
          'outputs/rail folder is available.')
else:
    if rail_summary is not None:
        print()
        for k in ['active_stations', 'undirected_service_edges', 'average_degree', 'density',
                  'connected_components', 'largest_component_share', 'average_clustering',
                  'average_shortest_path_hops_lcc', 'diameter_hops_lcc',
                  'articulation_points', 'bridges', 'louvain_communities']:
            if k in rail_summary:
                print(('  ' + k).ljust(38), fmt(rail_summary[k], 4))

        n_stations = get(rail_summary, 'active_stations')
        n_ap = get(rail_summary, 'articulation_points')
        n_br = get(rail_summary, 'bridges')
        n_edges = get(rail_summary, 'undirected_service_edges')
        record('11', 'rail', 'rail stations in the graph', n_stations, 'summary.json')
        record('11', 'rail', 'rail articulation points', n_ap, 'summary.json',
               '' if not (n_ap and n_stations) else
               format(100 * float(n_ap) / float(n_stations), '.1f') + '% of rail stations - '
               'far above the bus-network rate, because the rail graph is nearly a path')
        record('11', 'rail', 'rail bridges', n_br, 'summary.json',
               '' if not (n_br and n_edges) else
               format(100 * float(n_br) / float(n_edges), '.1f') + '% of rail segments')
        record('11', 'rail', 'rail network diameter (hops)',
               get(rail_summary, 'diameter_hops_lcc'), 'summary.json')

    if rail_damage is not None and len(rail_damage):
        sep_col = pick_col(rail_damage, 'stations_separated', 'separated_stations',
                           'components', 'largest_component_share_of_remaining')
        name_col = pick_col(rail_damage, 'stop_name')
        share_col = pick_col(rail_damage, 'largest_component_share_of_remaining')
        print()
        print('Exhaustive single-station closure - worst outcomes:')
        sort_col = share_col if share_col else sep_col
        if sort_col is None:
            print('  (no damage column recognised in single_station_damage.csv)')
            display(rail_damage.head(TOP_N))
        else:
            ascending = share_col is not None
            show = rail_damage.sort_values(sort_col, ascending=ascending).head(TOP_N)
            display(show.reset_index(drop=True))
        if share_col is not None:
            worst = rail_damage.loc[pd.to_numeric(rail_damage[share_col],
                                                  errors='coerce').idxmin()]
            worst_name = str(worst[name_col]) if name_col else ''
            record('11', 'rail', 'worst single-station closure (largest component left)',
                   fmt(worst[share_col], 4) + ' of the surviving stations',
                   'single_station_damage.csv',
                   ('station: ' + worst_name) if worst_name else '')

    if rail_metrics is not None:
        wd = pick_col(rail_metrics, 'weighted_degree')
        bt = pick_col(rail_metrics, 'betweenness')
        if wd and bt:
            rho_rail = float(pd.to_numeric(rail_metrics[wd], errors='coerce')
                             .corr(pd.to_numeric(rail_metrics[bt], errors='coerce'),
                                   method='spearman'))
            print()
            print('Rail: Spearman rho, weighted degree vs betweenness =',
                  format(rho_rail, '+.3f'))
            record('11', 'rail', 'rail Spearman rho (weighted degree vs betweenness)',
                   format(rho_rail, '+.3f'), 'rail_station_metrics.csv',
                   'compare with the national figure in Finding 1')

## 16. Finding 7 (negative) - the rail socioeconomic tests don't reach significance (stage 12)

Stage 12 links the rail stations to CBS socioeconomic data and runs a grid of Spearman correlations (two socioeconomic variables x
several network measures x two linking samples), then corrects for multiple comparisons with Benjamini-Hochberg and Holm.

**This is a null result and must be reported as one.** With only a few dozen rail stations - and even fewer once the linking is
restricted to stations falling inside a statistical-area polygon - the test has negligible statistical power, so "no significant
correlation" means *we couldn't detect an effect*, not *there is no effect*. The cell prints how many of the tests pass at the raw
alpha and how many survive correction, and reports the largest |rho| observed together with its uncorrected p-value, so the reader
can see the magnitude of what went undetected.


In [ ]:
# --- Rail socio-economic (negative result) --------------------------------
section_header('FINDING 7 (NEGATIVE) - RAIL SOCIO-ECONOMIC CORRELATIONS (stage 12)')

rail_corr = load_csv('12', 'rail_socioeconomic_correlations.csv')

if rail_corr is None:
    print('\nSKIPPED - stage 12 (rail socio-economic) has not been run.')
else:
    p_col = pick_col(rail_corr, 'p_value_two_sided', 'p_value')
    q_col = pick_col(rail_corr, 'q_value_bh')
    holm_col = pick_col(rail_corr, 'p_value_holm')
    rho_col = pick_col(rail_corr, 'spearman_rho', 'rho')
    n_col = pick_col(rail_corr, 'n_stations', 'n')

    print()
    display(rail_corr.round(4))

    n_tests = len(rail_corr)
    n_raw = int((pd.to_numeric(rail_corr[p_col], errors='coerce') < 0.05).sum()) if p_col else None
    n_bh = int((pd.to_numeric(rail_corr[q_col], errors='coerce') < 0.05).sum()) if q_col else None
    n_holm = int((pd.to_numeric(rail_corr[holm_col], errors='coerce') < 0.05).sum()) if holm_col else None

    print()
    print('tests run                  :', n_tests)
    print('raw p < 0.05               :', n_raw)
    print('BH q < 0.05                :', n_bh)
    print('Holm p < 0.05              :', n_holm)

    if rho_col:
        strongest = rail_corr.loc[pd.to_numeric(rail_corr[rho_col],
                                                errors='coerce').abs().idxmax()]
        metric_col = pick_col(rail_corr, 'network_metric')
        var_col = pick_col(rail_corr, 'socioeconomic_variable')
        detail = []
        if metric_col:
            detail.append(str(strongest[metric_col]))
        if var_col:
            detail.append(str(strongest[var_col]))
        if p_col:
            detail.append('raw p = ' + fmt(strongest[p_col], 3))
        print('largest |rho| observed     :', format(float(strongest[rho_col]), '+.3f'),
              '(' + ', '.join(detail) + ')' if detail else '')
        if n_col:
            print('sample size                :', fmt(strongest[n_col]), 'stations')
        record('12', 'rail equity', 'largest |rho| among the rail socio-economic tests',
               format(float(strongest[rho_col]), '+.3f'),
               'rail_socioeconomic_correlations.csv',
               'not significant after correction')

    record('12', 'rail equity', 'tests significant after BH-FDR',
           str(n_bh) + ' of ' + str(n_tests), 'rail_socioeconomic_correlations.csv',
           'NEGATIVE RESULT: underpowered at this sample size, not evidence of equality')

    print()
    print('HONEST READING: with this many stations the study is underpowered. A null result here '
          'is a failure to detect, not a demonstration that rail access is socio-economically '
          'neutral.')

## 17. Finding 8 (negative) - what graph learning *didn't* deliver (stage 13)

Stage 13 is the project's machine-learning arm, and it produced three negative results that are more informative than a positive
result would have been:

1. **Link prediction.** The original evaluation embedded the *full* graph and only then split the test pairs for the classifier,
   which lets information about the held-out edges leak into the embeddings. The corrected protocol embeds only the training graph.
   The cell prints both side by side, and the AUC difference between them - that difference is the size of the leakage, not a
   modeling result. It also reports the evaluation with hard negatives (pairs that aren't edges, at a distance of two hops), where
   the task becomes genuinely hard and the scores drop further.
2. **Adding the suggested edges.** The top suggested edges were added to the graph and the attack simulation was re-run. The
   measured change in resilience is reported exactly as computed - it's effectively zero, which is the expected result of adding a
   handful of edges to a graph that already has tens of thousands of them.
3. **Predicting critical stations from embeddings.** A classifier trained to identify stations in the top decile of betweenness
   from node2vec features reaches an F1 in a very low range, almost no different from guessing by the base rate - and in places
   even worse than it. The base-rate F1 is printed alongside so the comparison is explicit.


In [ ]:
# --- Embeddings, link prediction and the classifier (negative results) ----
section_header('FINDING 8 (NEGATIVE) - GRAPH LEARNING (stage 13)')

lp = load_csv('13', 'link_prediction_results.csv')
lp_hard = load_csv('13', 'link_prediction_hard_negatives.csv', quiet=True)
clf = load_csv('13', 'critical_classifier_results.csv')
res_imp = load_csv('13', 'resilience_improvement.csv')

if lp is None and clf is None and res_imp is None:
    print('\nSKIPPED - stage 13 (embeddings and link prediction) has not been run.')
else:
    # ---- 1. link prediction, leaked vs leakage-free
    if lp is not None and len(lp):
        m_col = pick_col(lp, 'method')
        p_col = pick_col(lp, 'protocol')
        a_col = pick_col(lp, 'auc')
        print()
        print('Link prediction results:')
        display(lp.sort_values([p_col, a_col], ascending=[True, False]).reset_index(drop=True)
                if p_col and a_col else lp)

        if m_col and p_col and a_col:
            n2v = lp[lp[m_col].astype(str).str.contains('node2vec', case=False)]
            leaked = n2v[n2v[p_col].astype(str).str.contains('leak', case=False)]
            clean = n2v[~n2v[p_col].astype(str).str.contains('leak', case=False)]
            if len(leaked) and len(clean):
                a_leak = float(leaked.iloc[0][a_col])
                a_clean = float(clean.iloc[0][a_col])
                print()
                print('node2vec AUC, leaked protocol      :', format(a_leak, '.4f'))
                print('node2vec AUC, leakage-free protocol:', format(a_clean, '.4f'))
                print('leakage inflation                  :', format(a_leak - a_clean, '+.4f'),
                      'AUC')
                record('13', 'graph learning (negative)',
                       'node2vec link-prediction AUC, leaked vs leakage-free',
                       format(a_leak, '.4f') + ' -> ' + format(a_clean, '.4f'),
                       'link_prediction_results.csv',
                       'the apparent advantage was largely train/test leakage')
            if len(clean):
                best_clean = lp[~lp[p_col].astype(str).str.contains('leak', case=False)]
                best_clean = best_clean.loc[pd.to_numeric(best_clean[a_col],
                                                          errors='coerce').idxmax()]
                print('best leakage-free method           :', str(best_clean[m_col]),
                      '(AUC =', format(float(best_clean[a_col]), '.4f') + ')')
                record('13', 'graph learning (negative)',
                       'best leakage-free link-prediction method',
                       str(best_clean[m_col]) + ' (AUC = '
                       + format(float(best_clean[a_col]), '.4f') + ')',
                       'link_prediction_results.csv',
                       'classical heuristics are competitive with the embeddings')

    if lp_hard is not None and len(lp_hard):
        print()
        print('Hard negatives (non-edges two hops apart) - the realistic version of the task:')
        display(lp_hard.round(4))

    # ---- 2. resilience improvement from the suggested links
    if res_imp is not None and len(res_imp):
        d_col = pick_col(res_imp, 'delta_pct')
        k_col = pick_col(res_imp, 'removal_k')
        print()
        print('Resilience after adding the top suggested links:')
        display(res_imp)
        if d_col:
            deltas = pd.to_numeric(res_imp[d_col], errors='coerce')
            print('largest improvement at any removal level:',
                  format(float(deltas.max()), '+.3f') + '%')
            print('mean improvement across removal levels  :',
                  format(float(deltas.mean()), '+.3f') + '%')
            record('13', 'graph learning (negative)',
                   'resilience gain from adding the suggested links',
                   format(float(deltas.max()), '+.1f') + '% (best case)',
                   'resilience_improvement.csv',
                   'NEGATIVE RESULT: adding a handful of edges to a graph of tens of thousands '
                   'changes nothing measurable')

    # ---- 3. critical-station classifier
    if clf is not None and len(clf):
        mo_col = pick_col(clf, 'model')
        f_col = pick_col(clf, 'f1_critical', 'f1')
        print()
        print('Critical-station classifier:')
        display(clf)
        if mo_col and f_col:
            base = clf[clf[mo_col].astype(str).str.contains('random|base', case=False)]
            learned = clf[~clf[mo_col].astype(str).str.contains('random|base', case=False)]
            if len(learned):
                f_vals = pd.to_numeric(learned[f_col], errors='coerce')
                best = learned.loc[f_vals.idxmax()]
                print()
                print('best learned model  :', str(best[mo_col]),
                      'F1 =', format(float(best[f_col]), '.4f'))
                print('F1 range across models:', format(float(f_vals.min()), '.4f'),
                      'to', format(float(f_vals.max()), '.4f'))
                if len(base):
                    print('random guess at the base rate: F1 =',
                          format(float(base.iloc[0][f_col]), '.4f'))
                record('13', 'graph learning (negative)',
                       'critical-station classifier F1 (range across models)',
                       format(float(f_vals.min()), '.3f') + ' - '
                       + format(float(f_vals.max()), '.3f'),
                       'critical_classifier_results.csv',
                       'NEGATIVE RESULT: barely distinguishable from guessing at the base rate'
                       + ('' if not len(base) else ' (F1 = '
                          + format(float(base.iloc[0][f_col]), '.3f') + ')'))

    print()
    print('HONEST READING: none of the three graph-learning results supports a claim that '
          'embeddings add value here. They are reported because a negative result obtained '
          'under a corrected protocol is worth more than a positive one obtained under a '
          'leaky one.')

## 18. Finding 9 - the six modes are six different networks (stages 14-16)

Everything measured in sections 7-17 was measured on a single graph that merges every `route_type` in the GTFS into one adjacency
structure. Stages 14-16 take that assumption apart: stage 14 inventories the modes and builds one graph per mode, stage 15 rebuilds
the bus network on its own, and stage 16 does the same for light rail, funicular, trolleybus and demand-responsive service, using
**exact** betweenness because these graphs are small enough for it.

The result is that "Israel's public transport network" isn't one object. The bus graph is a national network: a single giant
component holding almost every station, with cut nodes a small minority. The light-rail graph is the opposite regime - a few
disconnected lines, no cycles at all, a largest component holding only about a quarter of its stations, and almost every station a
cut node, because in a path graph every interior node is one. These two numbers don't describe the same kind of system and should
never be averaged together.

The cell also reads stage 15's bus-versus-all-modes comparison, which answers the natural follow-up: since buses are ~98% of the
trips, is the merged national graph simply the bus graph? The node coverage, the degree and betweenness rank correlations, and the
Jaccard index on the articulation points printed below tell you how close that identity is - and where it breaks.


In [ ]:
# --- Mode-by-mode comparison ----------------------------------------------
section_header('FINDING 9 - SIX MODES, SIX DIFFERENT NETWORKS (stages 14-16)')

mode_inv = load_csv('14', 'mode_inventory.csv')
mode_net = load_csv('14', 'mode_network_summary.csv')
mode_json = load_json('14', 'mode_comparison_summary.json', quiet=True)
bus_json = load_json('15', 'bus_summary.json')
minor_modes = load_csv('16', 'minor_modes_summary.csv')

LR_ROW = BUS_ROW = None

if mode_net is None and mode_inv is None and bus_json is None and minor_modes is None:
    print('\nSKIPPED - stage 14 (multimodal inventory) has not been run, so the per-mode '
          'comparison cannot be made. Run notebooks 14-16.')
else:
    # ---- (1) the inventory: how much of the feed each mode is
    if mode_inv is not None and len(mode_inv):
        lab = pick_col(mode_inv, 'mode_label', 'mode')
        trips = pick_col(mode_inv, 'trips')
        print()
        print('Modes in the feed:')
        print(mode_inv.to_string(index=False))
        record('14', 'multimodal', 'transport modes present in the feed', len(mode_inv),
               'mode_inventory.csv')
        if lab and trips:
            t = pd.to_numeric(mode_inv[trips], errors='coerce')
            total_trips = float(t.sum())
            if total_trips > 0:
                top = mode_inv.loc[t.idxmax()]
                record('14', 'multimodal', 'share of all scheduled trips run by the largest mode',
                       str(top[lab]) + ' ' + pct(float(top[trips]) / total_trips, 2),
                       'mode_inventory.csv',
                       'the merged national graph is dominated by this one mode')

    # ---- (2) one graph per mode: the structural contrast
    if mode_net is not None and len(mode_net):
        lab = pick_col(mode_net, 'mode_label', 'mode')
        n_col = pick_col(mode_net, 'nodes')
        e_col = pick_col(mode_net, 'edges', 'undirected_edges')
        c_col = pick_col(mode_net, 'components')
        lcc_col = pick_col(mode_net, 'largest_component_share')
        ap_col = pick_col(mode_net, 'articulation_points')
        print()
        print('One graph per mode:')
        print(mode_net.to_string(index=False))

        if lab and n_col:
            labels = mode_net[lab].astype(str)
            nodes_num = pd.to_numeric(mode_net[n_col], errors='coerce')
            BUS_ROW = mode_net.loc[nodes_num.idxmax()]          # the mesh: by far the largest
            lr_mask = (labels.str.contains('tram', case=False)
                       | labels.str.contains('light rail', case=False)) \
                & ~labels.str.contains('cable', case=False)
            if lr_mask.any():
                LR_ROW = mode_net[lr_mask].iloc[0]

            print()
            print('largest mode by stations :', str(BUS_ROW[lab]), '-',
                  fmt(BUS_ROW[n_col]), 'stations,',
                  'largest component', pct(get(BUS_ROW, lcc_col) if lcc_col else None, 2))
            if ap_col and n_col and float(BUS_ROW[n_col]):
                print('  cut vertices           :', fmt(BUS_ROW[ap_col]),
                      '(' + pct(float(BUS_ROW[ap_col]) / float(BUS_ROW[n_col]), 2)
                      + ' of its stations)')
                record('14', 'multimodal', 'cut-vertex share of the largest mode',
                       str(BUS_ROW[lab]) + ': '
                       + pct(float(BUS_ROW[ap_col]) / float(BUS_ROW[n_col]), 2),
                       'mode_network_summary.csv')
            if lcc_col:
                record('14', 'multimodal', 'largest-component share of the largest mode',
                       str(BUS_ROW[lab]) + ': ' + pct(BUS_ROW[lcc_col], 2),
                       'mode_network_summary.csv',
                       'a single national mesh - essentially every stop is reachable from every other')

            if LR_ROW is not None:
                print()
                print('light rail               :', fmt(LR_ROW[n_col]), 'stations,',
                      fmt(LR_ROW[e_col]) if e_col else 'n/a', 'segments,',
                      fmt(LR_ROW[c_col]) if c_col else 'n/a', 'components,',
                      'largest component', pct(get(LR_ROW, lcc_col) if lcc_col else None, 2))
                if ap_col and float(LR_ROW[n_col]):
                    print('  cut vertices           :', fmt(LR_ROW[ap_col]),
                          '(' + pct(float(LR_ROW[ap_col]) / float(LR_ROW[n_col]), 2)
                          + ' of its stations)')
                    record('14', 'multimodal', 'light-rail stations and cut vertices',
                           fmt(LR_ROW[n_col]) + ' stations, ' + fmt(LR_ROW[ap_col])
                           + ' cut vertices ('
                           + pct(float(LR_ROW[ap_col]) / float(LR_ROW[n_col]), 1) + ')',
                           'mode_network_summary.csv',
                           'a path/tree graph: almost every interior station is a single point of failure')
                if lcc_col:
                    record('14', 'multimodal', 'light-rail largest-component share',
                           pct(LR_ROW[lcc_col], 2), 'mode_network_summary.csv',
                           'disjoint lines - contrast with the bus mesh in the row above')
                if lcc_col and BUS_ROW is not None:
                    print()
                    print('CONTRAST: largest-component share is', pct(BUS_ROW[lcc_col], 2),
                          'for', str(BUS_ROW[lab]), 'and', pct(LR_ROW[lcc_col], 2),
                          'for', str(LR_ROW[lab]) + '. These are not the same kind of network, '
                          'and a national average over both is meaningless.')

    # ---- (3) shape of the minor modes (stage 16, exact centrality)
    if minor_modes is not None and len(minor_modes):
        shape_col = pick_col(minor_modes, 'shape_class')
        lab2 = pick_col(minor_modes, 'mode_label', 'mode')
        cyc_col = pick_col(minor_modes, 'cyclomatic_number')
        cols = [c for c in [lab2, pick_col(minor_modes, 'stops'),
                            pick_col(minor_modes, 'undirected_edges'),
                            pick_col(minor_modes, 'components'),
                            pick_col(minor_modes, 'largest_component_share'),
                            pick_col(minor_modes, 'articulation_point_share'),
                            cyc_col, shape_col,
                            pick_col(minor_modes, 'max_exact_betweenness')] if c]
        print()
        print('Minor modes measured with exact centrality (stage 16):')
        print(minor_modes[cols].to_string(index=False))
        if lab2 and shape_col:
            for _, r in minor_modes.iterrows():
                cyc_note = ''
                if cyc_col is not None:
                    cyc_val = pd.to_numeric(pd.Series([r[cyc_col]]), errors='coerce').iloc[0]
                    cyc_note = ('cyclomatic number = ' + fmt(cyc_val, 0)
                                + (' - no independent cycles at all, so no alternative route '
                                   'exists anywhere in this mode'
                                   if cyc_val == 0 else
                                   ' - independent cycles exist, so some segments can be '
                                   'routed around within the mode'))
                record('16', 'multimodal', 'shape of the ' + str(r[lab2]) + ' graph',
                       str(r[shape_col]), 'minor_modes_summary.csv', cyc_note)

    # ---- (4) is the merged national graph just the bus graph?
    if bus_json is not None:
        cov = jget(bus_json, 'bus_vs_all_mode')
        if isinstance(cov, dict):
            print()
            print('Is the merged all-mode graph just the bus graph? (stage 15)')
            for k in ['bus_nodes', 'all_mode_nodes', 'node_coverage_of_all_modes',
                      'undirected_edge_coverage_of_all_modes',
                      'spearman_degree_bus_vs_all_modes',
                      'spearman_betweenness_bus_vs_all_modes',
                      'top50_betweenness_overlap', 'articulation_point_jaccard',
                      'articulation_points_absent_from_bus']:
                if k in cov:
                    print(('  ' + k).ljust(46), fmt(cov[k], 4))
            record('15', 'multimodal', 'node coverage of the all-mode graph by the bus graph alone',
                   pct(cov.get('node_coverage_of_all_modes'), 2), 'bus_summary.json',
                   'the national graph is very nearly the bus graph')
            record('15', 'multimodal',
                   'Spearman rho, bus-only vs all-mode betweenness',
                   fmt(cov.get('spearman_betweenness_bus_vs_all_modes'), 4), 'bus_summary.json',
                   'high, but not 1 - the non-bus modes do move some stations')
            record('15', 'multimodal', 'cut vertices that exist only once other modes are added',
                   fmt(cov.get('articulation_points_absent_from_bus')), 'bus_summary.json',
                   'these single points of failure are invisible in a bus-only analysis')


## 19. Multimodal transfer hubs (stage 17)

Stage 17 is the only stage in the extended pipeline that is **optional**. It looks for stations where two or more modes meet - the
places where a bus passenger can actually reach the light rail or the train - and scores them as transfer hubs. If it wasn't run,
nothing downstream breaks: stage 23 simply reports that the multimodal lens isn't available and compares the remaining lenses.

So this section does one of two things. If stage 17's outputs exist, it prints how many hubs were found and their magnitude. If
they're missing, it says so explicitly - and so the reader sees the cost of the gap and not only its existence, it reads stage 23's
own record of which lenses were missing when the synthesis ran.


In [ ]:
# --- Multimodal transfer hubs (optional stage) -----------------------------
section_header('MULTIMODAL TRANSFER HUBS (stage 17 - optional)')

hubs = load_csv('17', 'transfer_hubs.csv', quiet=True)
hubs_json = load_json('17', 'multimodal_summary.json', quiet=True)

if hubs is None and hubs_json is None:
    print()
    print('SKIPPED - stage 17 (' + STAGE_FOLDERS['17'] + ') has not been run: no '
          'transfer_hubs.csv and no multimodal_summary.json under outputs/nb.')
    print('This stage is optional. Everything below still runs; the only consequence is that the '
          'multimodal lens is absent from the stage 23 lens comparison.')
    # show the consequence using stage 23's own bookkeeping, if stage 23 ran
    _lens_json_probe = load_json('23', 'lens_synthesis_summary.json', quiet=True)
    _missing = get(_lens_json_probe, 'lenses_missing', default=[]) or []
    if _missing:
        print()
        print('Lenses stage 23 recorded as unavailable when it ran:')
        for m in _missing:
            if isinstance(m, dict):
                print('   -', m.get('lens', '?'), '(needs', str(m.get('needs', '?')) + ':',
                      str(m.get('missing', '?')) + ')')
            else:
                print('   -', m)
        record('17', 'multimodal', 'lenses unavailable in the stage 23 synthesis',
               str(len(_missing)), 'lens_synthesis_summary.json',
               'stage 17 was not run, so the multimodal transfer lens could not be scored')
else:
    if hubs_json is not None:
        print()
        for k, v in (hubs_json.items() if isinstance(hubs_json, dict) else []):
            if not isinstance(v, (dict, list)):
                print(('  ' + str(k)).ljust(46), fmt(v, 4))
    if hubs is not None and len(hubs):
        print()
        print('transfer hubs found :', format(len(hubs), ','))
        modes_col = pick_col(hubs, 'n_modes', 'modes', 'mode_count')
        name_col = pick_col(hubs, 'stop_name', 'name')
        score_col = pick_col(hubs, 'hub_score', 'score', 'transfer_score')
        record('17', 'multimodal', 'multimodal transfer hubs identified', len(hubs),
               'transfer_hubs.csv')
        if modes_col is not None:
            m = pd.to_numeric(hubs[modes_col], errors='coerce')
            print('max modes at a single stop :', fmt(m.max()))
            record('17', 'multimodal', 'maximum number of modes meeting at one stop',
                   fmt(m.max()), 'transfer_hubs.csv')
        show_cols = [c for c in [pick_col(hubs, 'stop_id'), name_col, modes_col, score_col] if c]
        if show_cols:
            display(hubs[show_cols].head(TOP_N).reset_index(drop=True))


## 20. Finding 10 - hop count is the wrong distance measure, and it's a limitation of sections 7-17 (stage 18)

Every shortest path computed anywhere in stages 02-13 counts **hops**: each segment between two consecutive stations costs 1,
whether it's a 40-second crawl between two adjacent urban stops or a 20-minute intercity ride. Stage 18 rebuilds that same graph with
the **median scheduled travel time** of each segment as the edge weight, and asks the obvious question: does it matter?

It does, and the cell below quantifies it in three ways.

1. **Do the paths coincide?** Stage 18 samples station pairs, computes for each pair the hop-optimal path and the time-optimal path,
   and reports the share where the two agree. The share is negligible.
2. **What does hop routing cost a passenger?** The median number of seconds a passenger loses when they take the hop-optimal path
   instead of the time-optimal one, and the median ratio between the two durations. The time-optimal path is usually *longer in
   hops* - it includes more segments, each faster.
3. **Does it move the criticality ranking?** Betweenness is recomputed with travel-time weights and compared against a hop-count
   control **computed from the same pivot sample**, so the comparison isn't contaminated by sampling noise. Stage 18 also reports
   the noise floor directly: the same hop-count estimator against notebook 04's independent sample. If the correlation between
   travel-time and hop betweenness is much lower than this noise floor, the disagreement is a real property of the two measures.

This is reported as a **limitation of the earlier results**, not a correction to them: every "critical stations" list in sections
8-17 is a hop-count-based list, and stage 18 shows that a travel-time-based list would have been substantially different.


In [ ]:
# --- Travel time vs hop count ---------------------------------------------
section_header('FINDING 10 - TRAVEL TIME vs HOP COUNT (stage 18)')

tt_json = load_json('18', 'traveltime_summary.json')
tt_edges = load_csv('18', 'edges_traveltime.csv', quiet=True)
tt_paths = load_csv('18', 'path_comparison.csv', quiet=True)
tt_agree = load_csv('18', 'betweenness_agreement.csv')

if tt_json is None and tt_agree is None and tt_paths is None:
    print('\nSKIPPED - stage 18 (travel-time network) has not been run, so the hop-count '
          'assumption behind sections 8-17 is not quantified here.')
else:
    # ---- the weighted graph itself
    n_tt_edges = jget(tt_json, 'graph', 'undirected_edges')
    if n_tt_edges is None and tt_edges is not None:
        n_tt_edges = len(tt_edges)
    med_seconds = jget(tt_json, 'travel_time_seconds', 'p50')
    if med_seconds is None and tt_edges is not None:
        mcol = pick_col(tt_edges, 'median_travel_seconds')
        if mcol:
            med_seconds = float(pd.to_numeric(tt_edges[mcol], errors='coerce').median())
    print()
    print('segments carrying a measured travel time :', fmt(n_tt_edges))
    print('median segment travel time (seconds)     :', fmt(med_seconds, 1))
    print('segments left without a travel time      :',
          fmt(jget(tt_json, 'graph', 'stage02_segments_without_traveltime')))
    record('18', 'travel time', 'segments with a measured median travel time', n_tt_edges,
           'edges_traveltime.csv / traveltime_summary.json')
    record('18', 'travel time', 'median segment travel time (seconds)', fmt(med_seconds, 1),
           'traveltime_summary.json')

    # ---- (1) and (2): do hop-optimal and time-optimal routes coincide?
    pairs = jget(tt_json, 'path_comparison', 'pairs')
    same_share = jget(tt_json, 'path_comparison', 'identical_route_share')
    med_lost = jget(tt_json, 'path_comparison', 'median_seconds_lost_by_hop_routing')
    p90_lost = jget(tt_json, 'path_comparison', 'p90_seconds_lost_by_hop_routing')
    med_ratio = jget(tt_json, 'path_comparison', 'median_duration_ratio')
    med_extra_hops = jget(tt_json, 'path_comparison', 'median_extra_hops_on_time_route')
    if tt_paths is not None and len(tt_paths):
        same_col = pick_col(tt_paths, 'same_path')
        lost_col = pick_col(tt_paths, 'extra_seconds_if_hop_routed')
        ratio_col = pick_col(tt_paths, 'time_penalty_ratio')
        if pairs is None:
            pairs = len(tt_paths)
        if same_share is None and same_col:
            same_share = float(tt_paths[same_col].astype(str).str.lower()
                               .isin(['true', '1']).mean())
        if med_lost is None and lost_col:
            med_lost = float(pd.to_numeric(tt_paths[lost_col], errors='coerce').median())
        if med_ratio is None and ratio_col:
            med_ratio = float(pd.to_numeric(tt_paths[ratio_col], errors='coerce').median())

    print()
    print('station pairs compared                       :', fmt(pairs))
    print('pairs where the two routes are identical     :', pct(same_share, 2))
    print('median time lost by hop routing              :', fmt(med_lost, 1), 'seconds',
          '' if med_lost is None else '(' + fmt(float(med_lost) / 60.0, 1) + ' minutes)')
    print('90th-percentile time lost by hop routing     :', fmt(p90_lost, 1), 'seconds')
    print('median duration ratio (hop route / time route):', fmt(med_ratio, 3))
    print('median extra hops on the time-optimal route  :', fmt(med_extra_hops, 1))
    record('18', 'travel time (limitation)',
           'sampled station pairs where hop routing and time routing agree',
           pct(same_share, 2), 'traveltime_summary.json',
           'out of ' + fmt(pairs) + ' pairs - hop-optimal routes are almost never time-optimal')
    record('18', 'travel time (limitation)',
           'median passenger time lost by hop-count routing',
           fmt(med_lost, 1) + ' s'
           + ('' if med_lost is None else ' (' + fmt(float(med_lost) / 60.0, 1) + ' min)'),
           'traveltime_summary.json',
           'median duration ratio ' + fmt(med_ratio, 3)
           + ' - the hop route takes that many times longer')

    # ---- (3): does the criticality ranking move, beyond sampling noise?
    agree = tt_agree
    if agree is None:
        rows = jget(tt_json, 'betweenness', 'agreement')
        if isinstance(rows, list) and rows:
            agree = pd.DataFrame(rows)
    if agree is not None and len(agree):
        cmp_col = pick_col(agree, 'comparison')
        rho_col = pick_col(agree, 'spearman_all')
        t50_col = pick_col(agree, 'top50_overlap')
        print()
        print('Betweenness agreement between metrics:')
        print(agree.to_string(index=False))

        if cmp_col and rho_col:
            names = agree[cmp_col].astype(str)
            floor_mask = names.str.contains('noise floor', case=False)
            signal_mask = names.str.contains('hop control', case=False) & ~floor_mask
            if signal_mask.any() and floor_mask.any():
                sig = agree[signal_mask].iloc[0]
                flr = agree[floor_mask].iloc[0]
                rho_sig = float(pd.to_numeric(pd.Series([sig[rho_col]]), errors='coerce').iloc[0])
                rho_flr = float(pd.to_numeric(pd.Series([flr[rho_col]]), errors='coerce').iloc[0])
                print()
                print('travel-time vs hop, same pivot sample (signal) : rho =', format(rho_sig, '.4f'))
                print('hop vs hop, different pivot sample (noise floor): rho =',
                      format(rho_flr, '.4f'))
                print('gap                                            :',
                      format(rho_flr - rho_sig, '+.4f'))
                verdict = ('REAL SIGNAL: the two metrics disagree far more than the same-metric '
                           'resampling control does, so the difference is a property of the '
                           'distance definition, not of the sampling.'
                           if rho_flr - rho_sig > 0.05 else
                           'The gap to the noise floor is small; treat the metric difference '
                           'with caution.')
                print()
                print(verdict)
                record('18', 'travel time (limitation)',
                       'Spearman rho, travel-time vs hop-count betweenness (same sample)',
                       format(rho_sig, '.4f'), 'betweenness_agreement.csv',
                       'sampling-noise floor for the same estimator is '
                       + format(rho_flr, '.4f') + ' - the disagreement is real, not noise')
                if t50_col:
                    print()
                    print('top-50 overlap, travel-time vs hop :',
                          pct(pd.to_numeric(pd.Series([sig[t50_col]]),
                                            errors='coerce').iloc[0], 0))
                    print('top-50 overlap at the noise floor  :',
                          pct(pd.to_numeric(pd.Series([flr[t50_col]]),
                                            errors='coerce').iloc[0], 0))
                    record('18', 'travel time (limitation)',
                           'top-50 betweenness overlap, travel-time vs hop-count',
                           pct(pd.to_numeric(pd.Series([sig[t50_col]]),
                                             errors='coerce').iloc[0], 0),
                           'betweenness_agreement.csv',
                           'noise floor for the same comparison is '
                           + pct(pd.to_numeric(pd.Series([flr[t50_col]]),
                                               errors='coerce').iloc[0], 0))

    print()
    print('HONEST READING: sections 8-17 measure criticality in hops. Stage 18 shows that a '
          'travel-time definition produces a materially different ranking. The earlier results are '
          'not wrong, but they answer "which stations carry the most hop-shortest paths", not '
          '"which stations carry the most time-shortest journeys".')


## 21. Finding 11 - the network changes shape over the day, and so does criticality (stages 19-20)

The graph analyzed in sections 7-17 merges the whole service day into one static structure. Stage 19 splits the file into time
windows - morning peak, midday, afternoon peak, evening, night - and builds one graph per window from the trips actually running in
it. Stage 20 then re-runs the centrality measures and the attack simulation within each window.

Two things come out of this.

* **The network physically shrinks outside peak hours.** The night graph has noticeably fewer stations and segments than the daytime
  graphs. Thousands of stations simply have no night service, so any statement about night criticality is a statement about a
  smaller network, not the one in section 7.
* **Criticality rankings move between windows more than sampling noise.** Betweenness is a sampled estimate, so two windows will
  disagree a little even if nothing changed. Stage 20 measures that floor explicitly by running the estimator twice on the *same*
  window with different seeds. The cell compares the cross-window agreement against that within-window floor; when the cross-window
  agreement is clearly worse, the shift is real.

Fragility changes too: the cell reports which window is easiest to damage with targeted removal, as measured by the gap between the
area under a targeted strategy's curve and the random baseline in that window.


In [ ]:
# --- Time-of-day graphs and dynamic resilience -----------------------------
section_header('FINDING 11 - THE NETWORK BY TIME OF DAY (stages 19-20)')

windows = load_csv('19', 'window_summary.csv')
dyn_json = load_json('20', 'dynamic_summary.json')
dyn_curves = load_csv('20', 'dynamic_resilience.csv', quiet=True)
fragility = load_csv('20', 'fragility_summary.csv', quiet=True)
win_agree = load_csv('20', 'window_rank_agreement.csv', quiet=True)
noise_floor = load_csv('20', 'betweenness_noise_floor.csv', quiet=True)

if windows is None and dyn_json is None and dyn_curves is None:
    print('\nSKIPPED - stages 19-20 (time-of-day graphs and dynamic resilience) have not been '
          'run, so the static-snapshot assumption is not tested here.')
else:
    # ---- (1) the network shrinks off-peak
    if windows is not None and len(windows):
        w_col = pick_col(windows, 'window')
        n_col = pick_col(windows, 'nodes')
        e_col = pick_col(windows, 'undirected_edges', 'directed_edges')
        lcc_col = pick_col(windows, 'largest_component_share')
        print()
        print('One graph per time window (stage 19):')
        print(windows.to_string(index=False))
        if w_col and n_col:
            nodes_num = pd.to_numeric(windows[n_col], errors='coerce')
            biggest = windows.loc[nodes_num.idxmax()]
            smallest = windows.loc[nodes_num.idxmin()]
            shrink = 1.0 - float(smallest[n_col]) / float(biggest[n_col])
            print()
            print('busiest window :', str(biggest[w_col]), '-', fmt(biggest[n_col]), 'stations')
            print('thinnest window:', str(smallest[w_col]), '-', fmt(smallest[n_col]), 'stations')
            print('shrinkage      :', pct(shrink, 1), 'of stations lose all service')
            record('19', 'time of day', 'stations active in the thinnest service window',
                   str(smallest[w_col]) + ': ' + fmt(smallest[n_col]) + ' of '
                   + fmt(biggest[n_col]) + ' (' + pct(shrink, 1) + ' fewer)',
                   'window_summary.csv',
                   'off-peak criticality is measured on a materially smaller network')
            if lcc_col:
                record('19', 'time of day', 'largest-component share, busiest vs thinnest window',
                       pct(biggest[lcc_col], 2) + ' vs ' + pct(smallest[lcc_col], 2),
                       'window_summary.csv',
                       'the surviving night network stays connected - it is smaller, not broken')

    vanished = jget(dyn_json, 'movers', 'vanished_at_offpeak')
    if vanished is not None:
        print()
        print('stations present at peak but absent off-peak :', fmt(vanished),
              '(off-peak window:', str(get(dyn_json, 'offpeak_window', default='n/a')) + ')')
        record('20', 'time of day', 'stations that vanish entirely in the off-peak window',
               vanished, 'dynamic_summary.json',
               'they cannot be ranked off-peak at all, let alone protected')

    # ---- (2) do the rankings move more than the noise floor?
    rho_floor = t50_floor = None
    if noise_floor is not None and len(noise_floor):
        r = noise_floor.iloc[0]
        rho_floor = get(r, 'spearman_rho_same_window')
        t50_floor = get(r, 'top50_overlap_same_window')
    if rho_floor is None:
        rho_floor = jget(dyn_json, 'betweenness_noise_floor', 'spearman_rho_same_window')
        t50_floor = jget(dyn_json, 'betweenness_noise_floor', 'top50_overlap_same_window')

    if win_agree is not None and len(win_agree):
        m_col = pick_col(win_agree, 'metric')
        rho_col = pick_col(win_agree, 'spearman_rho')
        t50_col = pick_col(win_agree, 'top50_overlap')
        a_col = pick_col(win_agree, 'window_a')
        b_col = pick_col(win_agree, 'window_b')
        print()
        print('Cross-window rank agreement (stage 20):')
        print(win_agree.to_string(index=False))
        if m_col and rho_col:
            bt = win_agree[win_agree[m_col].astype(str).str.contains('betweenness', case=False)]
            if len(bt):
                rho_vals = pd.to_numeric(bt[rho_col], errors='coerce')
                worst = bt.loc[rho_vals.idxmin()]
                print()
                print('betweenness, worst-agreeing window pair :',
                      str(worst[a_col]) if a_col else '?', 'vs',
                      str(worst[b_col]) if b_col else '?',
                      '-> rho =', fmt(worst[rho_col], 4),
                      '' if not t50_col else '| top-50 overlap ' + pct(worst[t50_col], 0))
                print('same-window sampling noise floor        : rho =', fmt(rho_floor, 4),
                      '' if t50_floor is None else '| top-50 overlap ' + pct(t50_floor, 0))
                record('20', 'time of day',
                       'worst cross-window betweenness rank agreement',
                       'rho = ' + fmt(worst[rho_col], 4)
                       + ('' if not t50_col else ', top-50 overlap ' + pct(worst[t50_col], 0)),
                       'window_rank_agreement.csv',
                       'same-window noise floor is rho = ' + fmt(rho_floor, 4)
                       + ' - the movement between windows is larger than the sampling noise')
                try:
                    if rho_floor is not None and float(rho_floor) - float(worst[rho_col]) > 0.05:
                        print()
                        print('REAL MOVEMENT: cross-window disagreement is clearly worse than the '
                              'same-window resampling floor, so criticality genuinely depends on '
                              'the hour, not just on which pivots were sampled.')
                except (TypeError, ValueError):
                    pass

    med_shift = jget(dyn_json, 'movers', 'median_abs_rank_shift_in_pool')
    if med_shift is not None:
        print()
        print('median absolute rank shift, peak vs off-peak (top pool) :', fmt(med_shift),
              'positions, pool size', fmt(jget(dyn_json, 'movers', 'pool_size')))
        record('20', 'time of day', 'median rank shift between peak and off-peak',
               fmt(med_shift) + ' positions', 'dynamic_summary.json',
               'measured over the top pool of stations that exist in both windows')

    # ---- (3) which window is the most fragile?
    if fragility is not None and len(fragility):
        w_col = pick_col(fragility, 'window')
        s_col = pick_col(fragility, 'strategy')
        g_col = pick_col(fragility, 'targeting_gain_vs_random')
        auc_col = pick_col(fragility, 'auc')
        print()
        print('Attack simulation inside each window (stage 20):')
        print(fragility.to_string(index=False))
        if w_col and s_col and g_col:
            gains = pd.to_numeric(fragility[g_col], errors='coerce')
            worst = fragility.loc[gains.idxmax()]
            print()
            print('most damaging window/strategy combination :', str(worst[w_col]), '/',
                  str(worst[s_col]), '- targeting gain over random =', fmt(worst[g_col], 4),
                  '' if not auc_col else '(AUC ' + fmt(worst[auc_col], 4) + ')')
            record('20', 'time of day', 'window most exposed to targeted removal',
                   str(worst[w_col]) + ' / ' + str(worst[s_col]) + ' (gain over random = '
                   + fmt(worst[g_col], 4) + ')', 'fragility_summary.csv',
                   'targeting gain = random-baseline AUC minus this strategy AUC, same window')


## 22. Finding 12 - demand-weighted criticality, and why it's a proxy for population and not for ridership

Limitation 2 of this project has always been that GTFS contains no passengers. Stage 21 does the most that can honestly be done
about this without new data: it takes the **population** of CBS statistical areas, spreads it over the stations within each area
using a catchment kernel with distance decay, weights by scheduled service, and uses the result to weight the source nodes in the
betweenness computation. A station then scores high when it sits on paths that *people* might reasonably use, and not just on paths
that exist.

**This is a demand proxy, and it must never be cited as ridership.** It has no boardings, no alightings, no smartcard transfers and
no occupancy. It knows where people live, approximately, and where the schedule places service; it doesn't know where anyone
actually travels. The stage records this in its summary as `proxy_is_not_ridership`, which the cell prints verbatim so the caveat
travels with the number.

Because a weighted estimator differs from an unweighted one for two reasons - the weighting and the re-sampling - stage 21 also runs
a **uniform control**: the same estimator, the same sample size, the same family of seeds, uniform weights. The important comparison
is between the demand-weighted result and that control, not against notebook 04.


In [ ]:
# --- Demand-weighted criticality (population proxy, NOT ridership) ---------
section_header('FINDING 12 - DEMAND-WEIGHTED CRITICALITY, A POPULATION PROXY (stage 21)')

dem_json = load_json('21', 'demand_summary.json')
dem_rank = load_csv('21', 'demand_weighted_criticality.csv', dtype={'stop_id': str}, quiet=True)
dem_agree = load_csv('21', 'rank_agreement.csv')

if dem_json is None and dem_rank is None and dem_agree is None:
    print('\nSKIPPED - stage 21 (demand-weighted criticality) has not been run.')
else:
    # ---- the caveat first, read from the stage itself
    is_proxy = get(dem_json, 'proxy_is_not_ridership')
    print()
    print('proxy_is_not_ridership flag set by stage 21 :', str(is_proxy))
    desc = get(dem_json, 'proxy_description')
    if desc:
        print()
        print('How the weight is built:')
        print('  ' + str(desc))
    print()
    print('catchment radius (m)      :', fmt(jget(dem_json, 'parameters', 'catchment_radius_m')))
    print('decay length (m)          :', fmt(jget(dem_json, 'parameters', 'decay_length_m')))
    print('service exponent          :', fmt(jget(dem_json, 'parameters', 'service_exponent')))
    print('shortest-path definition  :', str(jget(dem_json, 'parameters', 'shortest_paths',
                                                  default='n/a')))
    record('21', 'demand proxy', 'demand weight is a population proxy, not ridership',
           'confirmed by the stage (proxy_is_not_ridership = ' + str(is_proxy) + ')',
           'demand_summary.json',
           'CBS population in a distance-decayed catchment x scheduled service - '
           'no boardings, no smart-card data, no occupancy anywhere in the project')

    # ---- coverage of the proxy
    pop_total = jget(dem_json, 'proxy_coverage', 'cbs_population_total')
    no_unit = jget(dem_json, 'proxy_coverage', 'stops_without_cbs_unit')
    zero_w = jget(dem_json, 'proxy_coverage', 'stops_with_zero_demand_weight')
    rho_pop_service = jget(dem_json, 'proxy_coverage',
                           'spearman_population_proxy_vs_service_volume')
    n_stops = jget(dem_json, 'network', 'stops')
    print()
    print('population allocated to stops :', fmt(pop_total))
    print('stops with no CBS unit        :', fmt(no_unit),
          '' if not (no_unit and n_stops) else '(' + pct(float(no_unit) / float(n_stops), 2) + ')')
    print('stops with zero demand weight :', fmt(zero_w))
    print('Spearman rho, population proxy vs scheduled service volume :',
          fmt(rho_pop_service, 4))
    record('21', 'demand proxy', 'stops that could not be matched to a CBS statistical area',
           fmt(no_unit) + (' of ' + fmt(n_stops) if n_stops else ''), 'demand_summary.json',
           'they receive no population weight, so their demand-weighted rank is a floor')
    record('21', 'demand proxy',
           'Spearman rho, population proxy vs scheduled service volume',
           fmt(rho_pop_service, 4), 'demand_summary.json',
           'weakly related - service is not placed where population is, or vice versa')

    # ---- does weighting by people change who is critical?
    if dem_agree is not None and len(dem_agree):
        print()
        print('Rank agreement between criticality lenses (stage 21):')
        print(dem_agree.to_string(index=False))
        a_col = pick_col(dem_agree, 'lens_a')
        b_col = pick_col(dem_agree, 'lens_b')
        r_col = pick_col(dem_agree, 'spearman_all_stations', 'spearman')
        t_col = pick_col(dem_agree, 'top50_overlap')
        if a_col and b_col and r_col:
            names = (dem_agree[a_col].astype(str) + ' | ' + dem_agree[b_col].astype(str))
            ctrl_mask = names.str.contains('control', case=False) \
                & names.str.contains('demand', case=False)
            base_mask = names.str.contains('control', case=False) \
                & names.str.contains('nb04', case=False)
            if ctrl_mask.any():
                row = dem_agree[ctrl_mask].iloc[0]
                print()
                print('demand-weighted vs uniform control : rho =', fmt(row[r_col], 4),
                      '' if not t_col else '| top-50 overlap ' + pct(row[t_col], 0))
                record('21', 'demand proxy',
                       'demand-weighted vs uniform-control betweenness ranking',
                       'rho = ' + fmt(row[r_col], 4)
                       + ('' if not t_col else ', top-50 overlap ' + pct(row[t_col], 0)),
                       'rank_agreement.csv',
                       'the control isolates the weighting from the resampling')
            if base_mask.any():
                row2 = dem_agree[base_mask].iloc[0]
                print('uniform control vs notebook 04     : rho =', fmt(row2[r_col], 4),
                      '' if not t_col else '| top-50 overlap ' + pct(row2[t_col], 0),
                      ' <- this is the sampling-noise reference')

    shift = jget(dem_json, 'agreement', 'median_abs_rank_shift_top2000')
    if shift is not None:
        print()
        print('median absolute rank shift among the top 2000 stations :', fmt(shift), 'positions')
        record('21', 'demand proxy', 'median rank shift when population is weighted in',
               fmt(shift) + ' positions', 'demand_summary.json',
               'measured over the top 2000 stations of the topological ranking')

    for key, label in [('biggest_gainer', 'largest gain in rank once population is weighted in'),
                       ('biggest_loser', 'largest loss in rank once population is weighted in')]:
        blob = get(dem_json, key)
        if isinstance(blob, dict):
            print()
            print(label + ':', str(blob.get('stop_name', '?')),
                  '(' + str(blob.get('stop_id', '?')) + ')',
                  '- topological rank', fmt(blob.get('topological_rank')),
                  '-> demand-weighted rank', fmt(blob.get('demand_weighted_rank')))
            record('21', 'demand proxy', label,
                   str(blob.get('stop_name', '?')) + ': #'
                   + fmt(blob.get('topological_rank')) + ' -> #'
                   + fmt(blob.get('demand_weighted_rank')), 'demand_summary.json',
                   'POPULATION PROXY - not observed ridership')

    if dem_rank is not None and len(dem_rank):
        cols = [c for c in [pick_col(dem_rank, 'stop_id'), pick_col(dem_rank, 'stop_name'),
                            pick_col(dem_rank, 'topological_rank'),
                            pick_col(dem_rank, 'demand_weighted_rank'),
                            pick_col(dem_rank, 'rank_shift')] if c]
        print()
        print('Top stations under the demand-weighted lens (population proxy, not ridership):')
        display(dem_rank[cols].head(TOP_N).reset_index(drop=True))

    print()
    print('WORDING RULE: call this "demand-weighted" or "population-weighted" criticality. Do not '
          'call it ridership-weighted. No dataset in this project observes a single passenger.')


## 23. Finding 13 - what a closure actually costs, in passenger-seconds (stage 22)

Sections 10 and 21 measure disruption structurally: does the network fragment, does the ranking move. Neither answers the question
an operator asks, which is "if I close this station tomorrow, how much worse does everyone's trip get?" Stage 22 answers it
directly. It takes the travel-time graph from stage 18, samples origin-destination pairs whose time-optimal path passes through a
candidate station, removes the station, and reroutes each affected pair. The cost of the closure is the added travel time summed
over the passengers it diverts, with an explicit penalty for pairs that become unreachable.

Two results emerge, and they point in opposite directions.

* **Almost nothing breaks, and the detours are short.** Only a handful of the tested pairs become unreachable at all, and the median
  detour is measured in tens of seconds. Israel's public transport is densely interwoven: there's almost always another way around.
* **But the ranking by passenger cost isn't the topological ranking.** The Spearman correlation between the passenger-time cost of a
  closure and the station's betweenness ranking is low, and the reported p-value says it can't be distinguished from no relationship
  at this number of candidate stations. The station whose closure costs the most passenger time isn't the station with the highest
  betweenness.

The second point is the one that matters for planning, and the cell reports its p-value honestly instead of citing the correlation
alone.


In [ ]:
# --- Rerouting: the passenger-time cost of a closure -----------------------
section_header('FINDING 13 - THE PASSENGER-TIME COST OF A CLOSURE (stage 22)')

rr_json = load_json('22', 'rerouting_summary.json')
rr = load_csv('22', 'rerouting_results.csv', dtype={'removed_stop': str}, quiet=True)

if rr_json is None and rr is None:
    print('\nSKIPPED - stage 22 (rerouting model) has not been run, so no passenger-time cost '
          'of a closure is available.')
else:
    n_candidates = jget(rr_json, 'constants', 'TOP_N_STATIONS')
    n_pairs = jget(rr_json, 'od_sample', 'pairs_sampled')
    through = jget(rr_json, 'od_sample', 'share_through_a_candidate')
    print()
    print('candidate stations closed one at a time :', fmt(n_candidates))
    print('origin-destination pairs sampled        :', fmt(n_pairs))
    print('share routed through a candidate        :', pct(through, 2))

    # ---- (1) how bad is a closure?
    med = jget(rr_json, 'detours_seconds', 'median')
    mean = jget(rr_json, 'detours_seconds', 'mean')
    p90 = jget(rr_json, 'detours_seconds', 'p90')
    under60 = jget(rr_json, 'detours_seconds', 'share_under_60s')
    disc = jget(rr_json, 'disconnection', 'total_pairs_disconnected')
    tested = jget(rr_json, 'disconnection', 'total_pairs_tested')
    print()
    print('median detour   :', fmt(med, 1), 'seconds')
    print('mean detour     :', fmt(mean, 1), 'seconds')
    print('p90 detour      :', fmt(p90, 1), 'seconds')
    print('detours under a minute :', pct(under60, 1), 'of rerouted pairs')
    print('pairs left unreachable :', fmt(disc), 'of', fmt(tested),
          '' if not (disc is not None and tested) else
          '(' + pct(float(disc) / float(tested), 3) + ')')
    record('22', 'rerouting', 'median detour after closing a critical station',
           fmt(med, 1) + ' s', 'rerouting_summary.json',
           'p90 = ' + fmt(p90, 1) + ' s - the network reroutes cheaply almost everywhere')
    if disc is not None and tested:
        record('22', 'rerouting', 'origin-destination pairs left unreachable by a closure',
               fmt(disc) + ' of ' + fmt(tested) + ' (' + pct(float(disc) / float(tested), 3) + ')',
               'rerouting_summary.json',
               'closing a single station essentially never disconnects anyone')

    # ---- (2) does passenger cost rank stations like topology does?
    rho = jget(rr_json, 'rank_agreement', 'spearman_rho')
    pval = jget(rr_json, 'rank_agreement', 'spearman_p_value')
    topk = jget(rr_json, 'rank_agreement', 'top_k')
    topk_ov = jget(rr_json, 'rank_agreement', 'top_k_overlap')
    max_shift = jget(rr_json, 'rank_agreement', 'max_abs_rank_shift')
    n_eval = jget(rr_json, 'rank_agreement', 'stations_evaluated')
    if rho is None and rr is not None:
        bt_col = pick_col(rr, 'approx_betweenness')
        cost_col = pick_col(rr, 'expected_cost_seconds')
        if bt_col and cost_col:
            rho = float(pd.to_numeric(rr[bt_col], errors='coerce')
                        .corr(pd.to_numeric(rr[cost_col], errors='coerce'), method='spearman'))
    print()
    print('Spearman rho, topological rank vs passenger-time rank :', fmt(rho, 4))
    print('p-value                                              :', fmt(pval, 6))
    print('stations evaluated                                   :', fmt(n_eval))
    if topk is not None and topk_ov is not None:
        print('top-' + fmt(topk) + ' overlap between the two rankings          :',
              fmt(topk_ov), 'of', fmt(topk))
    print('largest rank shift between the two orderings         :', fmt(max_shift), 'positions')
    record('22', 'rerouting',
           'Spearman rho, betweenness rank vs passenger-time cost rank',
           fmt(rho, 4), 'rerouting_summary.json',
           'p = ' + fmt(pval, 6) + ' over ' + fmt(n_eval) + ' candidate stations - '
           'topological criticality barely predicts what a closure costs passengers')
    if topk is not None and topk_ov is not None:
        record('22', 'rerouting', 'top-' + fmt(topk) + ' overlap, topological vs passenger-time',
               fmt(topk_ov) + ' of ' + fmt(topk), 'rerouting_summary.json')

    worst = get(rr_json, 'costliest_closure')
    if isinstance(worst, dict):
        print()
        print('costliest closure :', str(worst.get('stop_name', '?')),
              '(' + str(worst.get('stop_id', '?')) + ')',
              '- expected cost', fmt(worst.get('expected_cost_seconds'), 1), 'seconds,',
              'reachable share', pct(worst.get('reachable_share'), 2))
        record('22', 'rerouting', 'costliest single closure in passenger time',
               str(worst.get('stop_name', '?')) + ' ('
               + fmt(worst.get('expected_cost_seconds'), 1) + ' s expected cost)',
               'rerouting_summary.json',
               'expected cost includes the unreachability penalty of '
               + fmt(jget(rr_json, 'constants', 'DISCONNECT_PENALTY_S')) + ' s per lost pair')

    if rr is not None and len(rr):
        cols = [c for c in [pick_col(rr, 'removed_stop'), pick_col(rr, 'stop_name'),
                            pick_col(rr, 'median_detour_seconds'),
                            pick_col(rr, 'expected_cost_seconds'),
                            pick_col(rr, 'reachable_share'),
                            pick_col(rr, 'topological_rank'),
                            pick_col(rr, 'passenger_time_rank'),
                            pick_col(rr, 'rank_shift')] if c]
        print()
        print('Closures ordered by passenger-time cost:')
        display(rr[cols].head(TOP_N).reset_index(drop=True))


## 24. Finding 14 - "critical" has eight different meanings, and they barely agree (stage 23)

This is the project's central claim, and stage 23 is where it stops being an argument and becomes a measurement.

Stage 23 defines one lens per notion of criticality the pipeline can actually compute: structural (betweenness), operational
(scheduled service volume), topological severity (articulation points ranked by how many stations they disconnect), spatial (no
substitute within walking reach), demand-weighted (stage 21's population proxy), temporal (betweenness in the peak window from stage
20), passenger-time (closure cost from stage 22), and multimodal transfer importance (stage 17, optional). Each lens ranks the same
universe of stations. The stage then measures, for each pair of lenses, the Spearman rank correlation across all stations and the
overlap between the two top-50 lists.

The cell below reports:

* how many lenses were available and how many pairs were compared;
* the average, minimum and maximum pairwise rank correlation, and the average top-50 overlap;
* the **structural versus operational** pair specifically - the claim section 8 raises qualitatively, now with a tight top-50 overlap
  attached to it - and the pair with the weakest agreement of all;
* how many stations are flagged by any lens at all, how many of them are agreed on by two or more lenses, and how many are flagged
  by exactly one lens.

That last set of numbers is the punchline: if most flagged stations are flagged by exactly one lens, then a resilience plan built on
any single definition of criticality misses almost everything the other definitions consider important.


In [ ]:
# --- Eight lenses on "critical" -------------------------------------------
section_header('FINDING 14 - EIGHT LENSES ON "CRITICAL" (stage 23)')

lens_json = load_json('23', 'lens_synthesis_summary.json')
lens_agree = load_csv('23', 'lens_agreement.csv')
lens_cov = load_csv('23', 'lens_coverage.csv', quiet=True)
lens_rows = load_csv('23', 'critical_station_lenses.csv', dtype={'stop_id': str}, quiet=True)

LENS_TOPK = None

if lens_json is None and lens_agree is None and lens_rows is None:
    print('\nSKIPPED - stage 23 (critical-station lenses) has not been run, so the central '
          'multi-lens claim is not quantified here.')
else:
    # ---- which lenses exist, and how many stations each one can score
    available = get(lens_json, 'lenses_available', default=[]) or []
    missing = get(lens_json, 'lenses_missing', default=[]) or []
    LENS_TOPK = get(lens_json, 'top_k_definition', default=50)
    if not available and lens_rows is not None:
        lcol = pick_col(lens_rows, 'lens')
        available = sorted(lens_rows[lcol].astype(str).unique()) if lcol else []
    n_defined = len(available) + len(missing)
    print()
    print('lenses defined by stage 23 :', n_defined)
    print('lenses available on this run:', len(available), '->', ', '.join(map(str, available)))
    if missing:
        print('lenses unavailable          :', len(missing), '->',
              ', '.join(str(m.get('lens', m)) if isinstance(m, dict) else str(m)
                        for m in missing))
    record('23', 'lens synthesis', 'criticality lenses defined / available',
           str(len(available)) + ' of ' + str(n_defined), 'lens_synthesis_summary.json',
           '' if not missing else 'missing: '
           + ', '.join(str(m.get('lens', m)) if isinstance(m, dict) else str(m) for m in missing))

    if lens_cov is not None and len(lens_cov):
        print()
        print('What each lens asks, and how much of the network it can answer for:')
        cols = [c for c in [pick_col(lens_cov, 'lens'), pick_col(lens_cov, 'question'),
                            pick_col(lens_cov, 'metric'),
                            pick_col(lens_cov, 'stations_evaluated'),
                            pick_col(lens_cov, 'coverage_share_of_universe')] if c]
        with pd.option_context('display.max_colwidth', 80):
            display(lens_cov[cols])

    # ---- pairwise agreement between the lenses
    if lens_agree is not None and len(lens_agree):
        a_col = pick_col(lens_agree, 'lens_a')
        b_col = pick_col(lens_agree, 'lens_b')
        r_col = pick_col(lens_agree, 'spearman_rho')
        t_col = pick_col(lens_agree, 'top50_overlap')

        # top50_overlap is stored as a COUNT out of top_k_definition in this stage;
        # normalise to a share so it is comparable with the other sections.
        overlap_share = None
        if t_col:
            ov = pd.to_numeric(lens_agree[t_col], errors='coerce')
            k = float(LENS_TOPK) if LENS_TOPK else 50.0
            overlap_share = ov / k if float(ov.max(skipna=True)) > 1.0 else ov

        print()
        print('Pairwise agreement between lenses (top-' + fmt(LENS_TOPK) + ' overlap):')
        print(lens_agree.to_string(index=False))

        rho = pd.to_numeric(lens_agree[r_col], errors='coerce') if r_col else None
        print()
        print('lens pairs compared        :', len(lens_agree))
        if rho is not None:
            print('mean pairwise Spearman rho :', fmt(rho.mean(), 4))
            print('range                      :', fmt(rho.min(), 4), 'to', fmt(rho.max(), 4))
            record('23', 'lens synthesis', 'mean pairwise Spearman rho between criticality lenses',
                   fmt(rho.mean(), 4), 'lens_agreement.csv',
                   'over ' + str(len(lens_agree)) + ' lens pairs; range '
                   + fmt(rho.min(), 4) + ' to ' + fmt(rho.max(), 4)
                   + ' - the lenses are largely measuring different things')
        if overlap_share is not None:
            print('mean top-' + fmt(LENS_TOPK) + ' overlap        :',
                  pct(overlap_share.mean(), 1))
            record('23', 'lens synthesis',
                   'mean top-' + fmt(LENS_TOPK) + ' overlap between criticality lenses',
                   pct(overlap_share.mean(), 1), 'lens_agreement.csv')

        # the structural-vs-operational pair: the project's headline claim, quantified
        if a_col and b_col:
            pair_names = (lens_agree[a_col].astype(str) + ' | ' + lens_agree[b_col].astype(str))
            struct_ops = pair_names.str.contains('service_volume', case=False) & (
                pair_names.str.contains('betweenness', case=False)
                | pair_names.str.contains('articulation', case=False))
            if struct_ops.any():
                print()
                print('STRUCTURAL vs OPERATIONAL criticality - the project headline, quantified:')
                for idx in lens_agree[struct_ops].index:
                    row = lens_agree.loc[idx]
                    share = None if overlap_share is None else overlap_share.loc[idx]
                    print('  ' + str(row[a_col]) + ' vs ' + str(row[b_col]) + ' : rho =',
                          fmt(row[r_col], 4) if r_col else 'n/a',
                          '| top-' + fmt(LENS_TOPK) + ' overlap',
                          ('n/a' if share is None else
                           fmt(row[t_col], 0) + ' stations (' + pct(share, 0) + ')'))
                    record('23', 'lens synthesis',
                           'agreement: ' + str(row[a_col]) + ' vs ' + str(row[b_col]),
                           ('rho = ' + (fmt(row[r_col], 4) if r_col else 'n/a')
                            + ('' if share is None else
                               ', top-' + fmt(LENS_TOPK) + ' overlap ' + pct(share, 0))),
                           'lens_agreement.csv',
                           'structural vs operational criticality - near-zero overlap is the '
                           'central claim of the project')

            if overlap_share is not None:
                worst_idx = overlap_share.idxmin()
                wrow = lens_agree.loc[worst_idx]
                print()
                print('weakest-agreeing lens pair :', str(wrow[a_col]), 'vs', str(wrow[b_col]),
                      '-> rho =', fmt(wrow[r_col], 4) if r_col else 'n/a',
                      '| top-' + fmt(LENS_TOPK) + ' overlap', pct(overlap_share.loc[worst_idx], 0))
            if rho is not None and rho.notna().any():
                min_idx = rho.idxmin()
                mrow = lens_agree.loc[min_idx]
                print('most anti-correlated pair  :', str(mrow[a_col]), 'vs', str(mrow[b_col]),
                      '-> rho =', fmt(mrow[r_col], 4))
                record('23', 'lens synthesis', 'most anti-correlated lens pair',
                       str(mrow[a_col]) + ' vs ' + str(mrow[b_col]) + ' (rho = '
                       + fmt(mrow[r_col], 4) + ')', 'lens_agreement.csv',
                       'a negative correlation means the two lenses actively disagree about '
                       'which stations matter')

    # ---- how much do the lenses actually overlap on stations?
    flagged_any = get(lens_json, 'stations_flagged_by_any_lens')
    flagged_2 = get(lens_json, 'stations_flagged_by_two_or_more')
    flagged_3 = get(lens_json, 'stations_flagged_by_three_or_more')
    flagged_1 = get(lens_json, 'stations_flagged_by_exactly_one')
    max_agree = get(lens_json, 'max_lenses_agreeing_on_one_station')
    if flagged_any is None and lens_rows is not None:
        id_col = pick_col(lens_rows, 'stop_id')
        rank_col = pick_col(lens_rows, 'rank')
        lcol = pick_col(lens_rows, 'lens')
        if id_col and rank_col and lcol and LENS_TOPK:
            top = lens_rows[pd.to_numeric(lens_rows[rank_col], errors='coerce')
                            <= float(LENS_TOPK)]
            counts = top.groupby(id_col)[lcol].nunique()
            flagged_any = int(len(counts))
            flagged_1 = int((counts == 1).sum())
            flagged_2 = int((counts >= 2).sum())
            flagged_3 = int((counts >= 3).sum())
            max_agree = int(counts.max()) if len(counts) else None

    if flagged_any:
        print()
        print('stations in at least one lens top-' + fmt(LENS_TOPK) + ' :', fmt(flagged_any))
        print('flagged by exactly one lens                :', fmt(flagged_1),
              '' if not (flagged_1 and flagged_any) else
              '(' + pct(float(flagged_1) / float(flagged_any), 1) + ')')
        print('flagged by two or more lenses              :', fmt(flagged_2))
        print('flagged by three or more lenses            :', fmt(flagged_3))
        print('most lenses agreeing on a single station   :', fmt(max_agree),
              'of', fmt(len(available)) if available else 'n/a')
        record('23', 'lens synthesis', 'stations flagged critical by at least one lens',
               flagged_any, 'lens_synthesis_summary.json',
               'out of ' + fmt(get(lens_json, 'stations_in_universe')) + ' stations')
        record('23', 'lens synthesis', 'stations flagged by exactly one lens',
               fmt(flagged_1) + (' (' + pct(float(flagged_1) / float(flagged_any), 1) + ')'
                                 if flagged_1 else ''),
               'lens_synthesis_summary.json',
               'a programme built on one definition of criticality misses the rest')
        record('23', 'lens synthesis', 'stations two or more lenses agree on',
               flagged_2, 'lens_synthesis_summary.json',
               'these are the defensible shortlist candidates')
        if max_agree is not None:
            record('23', 'lens synthesis', 'most lenses ever agreeing on one station',
                   fmt(max_agree) + ' of ' + (fmt(len(available)) if available else 'n/a'),
                   'lens_synthesis_summary.json',
                   'no station is critical under every definition')

    shortlist_n = get(lens_json, 'shortlist_size')
    if shortlist_n is not None:
        print()
        print('consensus shortlist size :', fmt(shortlist_n))
        record('23', 'lens synthesis', 'consensus critical-station shortlist',
               shortlist_n, 'critical_station_shortlist.csv',
               'stations ranked by how many lenses place them near the top')

    print()
    print('CENTRAL CLAIM, QUANTIFIED: "critical station" is not a property of a station. It is a '
          'property of a station AND a definition. The lenses above disagree with each other far '
          'more than they agree, so any resilience programme must state which definition it is '
          'optimising - and accept that it is not covering the others.')


## 25. The cross-stage summary table

Every number recorded in the sections above, in one table, together with the file it came from. Rows appear only for stages that
actually ran, so this table also serves as a record of what the pipeline produced on this machine. It's saved to
`tables/headline_findings.csv`.


In [ ]:
# --- Headline findings table ----------------------------------------------
section_header('HEADLINE FINDINGS')

if not FINDINGS:
    print('No findings were recorded - none of the upstream stages produced readable output. '
          'Run notebooks 00-23 first.')
    headlines = pd.DataFrame(columns=['stage', 'theme', 'metric', 'value', 'source_file', 'note'])
else:
    headlines = pd.DataFrame(FINDINGS)

headlines.to_csv(TABLES / 'headline_findings.csv', index=False, encoding='utf-8-sig')
print('rows:', len(headlines))
print('saved ->', TABLES / 'headline_findings.csv')
print()
if len(headlines):
    with pd.option_context('display.max_colwidth', 90, 'display.max_rows', 200):
        display(headlines)


### Figure 2 - the project at a glance

A single four-panel figure summarizing the whole study. Each panel is drawn only if its stage ran; otherwise the panel prints a
short "not available" message in place of the graph, so the figure is always produced and always honest about what's missing.

* **(a) Resilience** - the largest component's share versus the number of stations removed, targeted strategies versus the random
  baseline, using stage 06's surviving-normalized measure (the conservative one of the two).
* **(b) Substitutability** - stage 05's distance bands, green where a critical station has an alternative within a short walk and red
  where it doesn't.
* **(c) Regional criticality** - each region's share of stations falling in the national top decile of betweenness, with the
  national average as a reference line.
* **(d) The negative results** - link-prediction AUC under the leaky protocol and under the leak-free protocol against the chance
  line at 0.5, and the classifier's F1 against base-rate guessing. This panel exists so the negative findings are as visible as the
  positive ones.


In [ ]:
# --- Figure 2: project at a glance ----------------------------------------
def _unavailable(ax, message):
    """Blank out a panel with an explanation instead of dropping it silently."""
    ax.text(0.5, 0.5, message, ha='center', va='center', fontsize=11,
            color='#64748b', wrap=True, transform=ax.transAxes)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.grid(False)


fig, axes = plt.subplots(2, 2, figsize=(15, 11))

# ---- (a) resilience curves
ax = axes[0][0]
if disruption is None or not len(disruption):
    _unavailable(ax, '(a) Resilience curves\n\nstage 06 not available')
else:
    s_col = pick_col(disruption, 'strategy')
    r_col = pick_col(disruption, 'removed')
    y_col = pick_col(disruption, 'lcc_share_surviving', 'lcc_share_original', 'lcc_share')
    palette = {'degree': '#dc2626', 'weighted degree': '#ea580c', 'pagerank': '#16a34a',
               'betweenness': '#2563eb', 'articulation points': '#7c3aed',
               'random (baseline)': '#6b7280'}
    for strat, grp in disruption.groupby(s_col):
        grp = grp.sort_values(r_col)
        is_random = 'random' in str(strat).lower()
        ax.plot(pd.to_numeric(grp[r_col], errors='coerce'),
                pd.to_numeric(grp[y_col], errors='coerce'),
                label=str(strat), linewidth=2.0 if is_random else 1.6,
                linestyle='--' if is_random else '-',
                color=palette.get(str(strat), None), marker='o', markersize=2.6)
    ax.set_xlabel('stations removed')
    ax.set_ylabel(y_col)
    ax.set_title('(a) Robust to random failure, fragile to targeted attack')
    ax.legend(fontsize=8, loc='lower left')

# ---- (b) substitutability
ax = axes[0][1]
if iso_summary is None or not len(iso_summary):
    _unavailable(ax, '(b) Substitutability of critical stations\n\nstage 05 not available')
else:
    b_col = pick_col(iso_summary, 'distance_band')
    n_col = pick_col(iso_summary, 'n_critical_stops', 'n_stops', 'count')
    v_col = pick_col(iso_summary, 'verdict')
    counts = pd.to_numeric(iso_summary[n_col], errors='coerce').fillna(0)
    if v_col is not None:
        colors = ['#dc2626' if 'isolat' in str(v).lower() else '#16a34a'
                  for v in iso_summary[v_col]]
    else:
        colors = ['#2563eb'] * len(iso_summary)
    bars = ax.bar(iso_summary[b_col].astype(str), counts, color=colors, edgecolor='white')
    total = float(counts.sum()) or 1.0
    for bar, v in zip(bars, counts):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
                format(int(v), ',') + '\n(' + format(100 * v / total, '.0f') + '%)',
                ha='center', va='bottom', fontsize=9)
    ax.margins(y=0.20)
    ax.set_xlabel('distance to the nearest alternative stop')
    ax.set_ylabel('critical stations')
    ax.set_title('(b) Green = substitutable on foot, red = genuine single point of failure')

# ---- (c) regional criticality
ax = axes[1][0]
if regional is None or not len(regional):
    _unavailable(ax, '(c) Criticality by region\n\nstage 07 not available')
else:
    reg_col = pick_col(regional, 'region')
    pct_col = pick_col(regional, 'pct_critical')
    tot_col = pick_col(regional, 'total_stops')
    crit_col = pick_col(regional, 'critical_stops')
    order = regional.sort_values(pct_col, ascending=False)
    bars = ax.bar(order[reg_col].astype(str),
                  pd.to_numeric(order[pct_col], errors='coerce'),
                  color='#0891b2', edgecolor='white')
    if tot_col and crit_col:
        nat = (100 * pd.to_numeric(regional[crit_col], errors='coerce').sum()
               / max(1.0, float(pd.to_numeric(regional[tot_col], errors='coerce').sum())))
        ax.axhline(nat, color='#334155', ls='--', lw=1.2,
                   label='national average (' + format(nat, '.1f') + '%)')
        ax.legend(fontsize=9)
    if tot_col:
        for bar, n in zip(bars, pd.to_numeric(order[tot_col], errors='coerce')):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
                    'n=' + format(int(n), ','), ha='center', va='bottom', fontsize=8)
    ax.margins(y=0.20)
    ax.set_ylabel('% of the region\'s own stops that are critical')
    ax.set_title('(c) Statistically significant, substantively small')

# ---- (d) the negative results
ax = axes[1][1]
neg_labels, neg_values, neg_colors = [], [], []
if lp is not None and len(lp):
    m_col = pick_col(lp, 'method')
    p_col = pick_col(lp, 'protocol')
    a_col = pick_col(lp, 'auc')
    if m_col and p_col and a_col:
        n2v = lp[lp[m_col].astype(str).str.contains('node2vec', case=False)]
        for _, r in n2v.iterrows():
            leaked = 'leak' in str(r[p_col]).lower()
            neg_labels.append('node2vec AUC\n(' + str(r[p_col]) + ')')
            neg_values.append(float(r[a_col]))
            neg_colors.append('#dc2626' if leaked else '#2563eb')
if clf is not None and len(clf):
    mo_col = pick_col(clf, 'model')
    f_col = pick_col(clf, 'f1_critical', 'f1')
    if mo_col and f_col:
        for _, r in clf.iterrows():
            is_base = bool(re.search('random|base', str(r[mo_col]), re.I))
            neg_labels.append(str(r[mo_col])[:28] + '\n(F1)')
            neg_values.append(float(r[f_col]))
            neg_colors.append('#94a3b8' if is_base else '#7c3aed')

if not neg_labels:
    _unavailable(ax, '(d) Negative results\n\nstage 13 not available')
else:
    y = np.arange(len(neg_labels))
    ax.barh(y, neg_values, color=neg_colors, edgecolor='white')
    for yi, v in zip(y, neg_values):
        ax.text(v + 0.012, yi, format(v, '.3f'), va='center', fontsize=9)
    ax.axvline(0.5, color='#334155', ls='--', lw=1.0, label='AUC chance level (0.5)')
    ax.set_yticks(y)
    ax.set_yticklabels(neg_labels, fontsize=8)
    ax.invert_yaxis()
    ax.set_xlim(0, 1.15)
    ax.set_xlabel('score (AUC for link prediction, F1 for the classifier)')
    ax.set_title('(d) The negative results, at the same scale')
    ax.legend(fontsize=8, loc='lower right')

fig.suptitle('Israel public transport network - the project at a glance', fontsize=15)
fig.tight_layout(rect=(0, 0, 1, 0.97))
fig.savefig(FIGURES / 'project_at_a_glance.png', dpi=FIG_DPI, bbox_inches='tight')
plt.show()
print('saved ->', FIGURES / 'project_at_a_glance.png')

## 26. Synthesis - what the project actually found

Read this together with the printed numbers above; nothing here is a number this notebook invented.

**1. Operational criticality and structural criticality are two different problems, and network managers only see the first.**
Weighted degree - the number of scheduled trip records touching a station - is what an operator naturally monitors, and it's
dominated by dense urban interchanges where many lines overlap. Betweenness and articulation-point status are dominated by the
sparse connecting tissue: the single station linking a town to the trunk network, the single segment crossing a valley. The
Spearman correlation between the two is far below 1 and the top-K overlap is small at every depth measured, so a protection budget
spent on the busiest stations leaves most structurally critical stations uncovered. This is the project's headline, and it's a
planning claim rather than a mathematical curiosity.

**2. The network is resilient to random failure and fragile under targeted removal - but the size of that fragility depends on the
normalization, and the honest version is the smaller one.** Removing stations at random barely dents the largest connected
component; removing them by centrality order degrades it much faster. This is the expected behavior of a heavy-tailed spatial
network. The caveat matters: if the largest component is divided by the *original* station count, the curve drops simply because
stations were deleted, with no fragmentation at all. Stage 06 exports both normalizations and this notebook cites the
surviving-normalized one, under which the collapse is real but considerably less dramatic than the original report implied.

**3. Single failure points are a genuine minority phenomenon, and most of them have no substitute.** Articulation points are a small
percent of the stations, but the spatial substitution test shows that a large share of the highest-betweenness stations have no
other station within walking distance. The caveat is severe and recurs in the limitations: a nearby station is a substitute only if
it's served by a useful line, and that was never checked.

**4. Equity: the answer depends on the level of aggregation, which means there is no single answer.** At the national level,
per-capita service and a neighborhood's socioeconomic cluster correlate with one sign; within individual cities the correlations
point the other way and almost none of them survive correction for multiple comparisons. This is a Simpson's paradox and it's the
most methodologically important result in the project: it rules out any headline of the form "Israeli public transport shortchanges
weak neighborhoods" *or* its inverse, when that headline is derived from a single national correlation.

**5. The regional differences are statistically significant and substantively negligible.** With tens of thousands of stations,
chi-square finds everything significant; Cramer's V places the relationship in the negligible-to-small band and the gap between the
extreme regions is a few percentage points. The correct conclusion is "criticality isn't concentrated in one region", not "region X
is at risk".

**6. The rail sub-network is a different regime and should never be averaged into the bus network.** It's smaller by two orders of
magnitude, close to a path graph, and as a result a large share of its stations are cut nodes - structural fragility is the norm
there, not the exception. The exhaustive single-station removal (feasible only at this scale) confirms individual stations whose
loss splits the system. But with only a few dozen stations, its socioeconomic tests have almost no statistical power and didn't
reach significance at all.

**7. The network is indeed a collection of loosely coupled regional clusters - but their number shouldn't be cited.** Louvain finds
a high-modularity partition whose communities map recognizably onto metropolitan and regional geography, and only a small share of
the edges cross a community boundary. The *number* of communities, however, moves between random seeds even though the partition
itself stays almost identical, so the honest statement is the range printed in section 13 together with the modularity, and never a
single count.

**8. Graph learning added nothing here, and that's a result.** Under a leak-free protocol, node2vec's link-prediction advantage
mostly evaporates and classical heuristics are competitive; adding the top suggested edges changed simulated resilience by
effectively zero; and an embedding-based classifier for critical stations reached an F1 close to base-rate guessing. Reported as
failures, these tell the reader something real about the problem: criticality here is a *global* path property that local-adjacency
embeddings can't capture.
**9. "The network" is six networks, and only one of them is a connected fabric.** Stage 14 builds a graph per `route_type` in the
GTFS. The bus graph is a single national component covering almost every station, with cut nodes a small minority. The light-rail
graph, on the order of a hundred stations, has no cycles at all: it's a collection of disconnected paths and trees, its largest
component holds about a quarter of its stations, and almost every station in it is a cut node - and that's not a finding that light
rail is fragile but a statement that a path graph has no alternative routes by construction. Stage 15 also shows that the merged
national graph is very close to the bus graph alone; the value of the other modes isn't that they change the aggregate, but that
they contribute cut nodes that a bus-only analysis can't see.

**10. Hop count is the wrong distance measure, and it's a limitation of everything above section 18.** When the edges carry median
scheduled travel time instead of a uniform cost of 1, hop-optimal and time-optimal paths almost never coincide, and a passenger who
takes the hop-optimal path loses a large median amount of time - the time-optimal path usually includes *more* hops, because they're
faster. The betweenness rankings under the two measures correlate far below the level the same estimator reaches against an
independent re-sample of itself, so this is a real difference between the measures and not sampling noise. The criticality lists in
sections 8-17 remain valid answers to the hop-count question; they aren't the answer to the travel-time question.

**11. The network isn't one static object: it shrinks at night and its criticality ranking moves with the hour.** Stage 19's window
graphs lose thousands of stations outside peak hours, and thousands more stations that exist at peak have no service at all at night.
Stage 20 shows that the betweenness rankings move between windows more than the re-sampling floor within the same window, so the
shift is real. A "critical station" is therefore also a function of when you ask.

**12. Weighting by population changes who's critical - and it's still not ridership.** Stage 21 spreads CBS population over the
stations using a catchment with distance decay, weights by scheduled service, and re-weights the betweenness sources. Compared
against a uniform control that isolates the weighting from the re-sampling, the ranking moves meaningfully. This is the closest to
demand the project reaches, and it must be labeled a **population proxy**: this data has no boardings, no smartcard transfers and no
occupancy anywhere.

**13. It's easy to route around a closure, but the cheapest closures aren't necessarily the topologically unimportant ones.** Stage
22 removes candidate stations from the travel-time graph and reroutes the affected origin-destination pairs. Almost nothing becomes
unreachable and the median detour is small, which is a genuinely good property of a dense fabric. But the ranking of stations by
passenger-time cost correlates only weakly with the betweenness ranking, at a p-value that doesn't distinguish it from no
relationship - so betweenness isn't a usable predictor of what a closure costs passengers.

**14. And therefore: "critical" isn't a property of a station, but a property of a station and a definition.** Stage 23 scores every
station under eight independent lenses and measures how much they agree. The average pairwise rank correlation is low, some of the
pairs are genuinely negatively correlated, structural and operational criticality share almost nothing in their top-50 lists, and
the large majority of flagged stations are flagged by exactly one lens. This is the project's central claim, and now it's a
measurement rather than an argument: a resilience plan has to state which definition of criticality it's optimizing, because
optimizing one doesn't cover the others.


## 27. Limitations

These aren't caveats for the sake of caveats. Each one is a specific way in which a number above is narrower than it looks, and each
one could change a conclusion.

1. **Betweenness is sampled and approximate.** Exact betweenness on a graph of this size is too expensive to compute, so stage 04
   estimates it from a few hundred random source nodes. A large share of stations get an estimate of exactly zero simply because no
   sampled path passed through them - that's a measurement artifact, not a statement about the station. Stage 04's two-seed
   stability check quantifies the noise (printed in section 8): the top of the ranking is reasonably stable, individual
   mid-ranking stations are not. Every "critical stations" list in this project inherits this uncertainty, including the top-decile
   threshold that stages 05 and 07 use.
2. **Scheduled service isn't observed ridership - including in stage 21.** The edge weights and the "operational criticality" come
   from the GTFS schedule, that is, from what the operators *plan to run*. No passenger counts, smartcard transfers or occupancy
   data enter the project at any stage. A station with heavy scheduled service and empty buses looks the same here as a busy one.
   Any claim about "the busiest stations" is a claim about supply, never about demand. Stage 21's demand-weighted criticality does
   **not** remove this limitation: it's CBS residential population spread over a distance-decay catchment and multiplied by
   scheduled service. It knows approximately where people live and where the schedule places buses; it doesn't know where anyone
   travels. It should be called population-weighted, never ridership-weighted (see limitation 14).
3. **The edge weights count GTFS trip records, not service days.** A weight is the number of trip rows crossing a segment in the
   file, which conflates frequency with how the agency happens to encode its calendar: an operator that files one trip per calendar
   date produces far higher weights than an operator that files a single trip with a service pattern covering the same dates.
   Weighted-degree comparisons *across agencies* are therefore unsafe, and the national weighted-degree ranking partly reflects
   file-authoring conventions.
4. **No consolidation to parent stations - platforms are split.** Every `stop_id` in the GTFS is a node in its own right. A large
   interchange served by several platforms or bays appears as several separate low-degree nodes instead of one high-degree hub,
   which deflates the degree and betweenness of exactly the stations that matter most, and can create spurious articulation points
   where one platform is the only encoded link to another. Consolidating by `parent_station` is the highest-value fix available,
   and it's the first item in future work.
5. **The region assignment is a crude latitude-longitude rule.** Stations are tagged by coordinate bands, not by an official
   municipal or district boundary layer. Stations near a boundary are misassigned, and the regional comparison in section 11 is
   only as good as that rule. The socioeconomic link in stage 09 is done properly against CBS statistical-area polygons - the two
   shouldn't be confused, and the polygon-based numbers should be preferred wherever both exist.
6. **The substitution test is spatial only.** Stage 05 asks whether another *station* exists within walking distance. It never
   checks whether that station is served by a line reaching any useful destination, at a useful frequency, at the relevant hour.
   Two stations on opposite sides of a road served by the same single line count as substitutes; a station 100 m away served only
   by a school shuttle counts as a substitute. The "substitutable" share is therefore an **upper bound** on true substitutability,
   and the "isolated" share is a lower bound on the true number of single failure points.
7. **Hop-count shortest paths are the wrong distance measure - and stage 18 now says by how much.** Every path in stages 02-13
   costs 1 per segment, whether that segment is 40 seconds or 20 minutes. Stage 18 rebuilds the graph with median scheduled travel
   time on each edge and finds that hop-optimal and time-optimal paths coincide on a negligible share of the sampled station pairs,
   that hop routing costs a large median time penalty, and that betweenness under the two measures correlates far below the
   re-sampling floor within the same measure. This is a **quantified limitation of every criticality list in sections 8-17**, not a
   footnote: those lists answer "which stations carry the most hop-short paths", which is a different question from "which stations
   carry the most time-short trips". Waiting time, transfer penalties and service headways are still absent even from stage 18,
   whose weights are in-vehicle travel time only. "Disconnected" accordingly means topologically disconnected, not "unreachable in
   reasonable time".
8. **Simulated attacks aren't real disruptions.** Stage 06 deletes nodes instantly and measures the largest connected component.
   Real failures are partial, local, and met with a response - rerouting, replacement service, passengers walking. The largest
   component's share is a structural proxy for loss of accessibility, not a forecast of it.
9. **Small samples in the rail analysis.** With a few dozen stations, the rail socioeconomic tests (stage 12) have no statistical
   power by construction; failing to reject the null there says almost nothing.
10. **Multiple comparisons.** Stages 08 and 12 run families of correlations. Corrected p-values (BH, Holm/Bonferroni) are reported
    and are the ones to cite; raw significances within these families aren't evidence on their own.
11. **Community counts are seed-dependent.** Louvain is a stochastic greedy heuristic; stage 08's seed and resolution sweeps show
    that the community count moves between runs even though the partition is almost identical each time (high pairwise ARI). Cite
    the modularity and the count range, never a single community count, and never compare a count from this project against a count
    from another study that used a different resolution parameter.
12. **One file, one point in time.** The whole project rests on a single GTFS snapshot. Seasonal variation, service changes and
    errors in the file aren't visible, and nothing here was validated against a second version.
13. **Scheduled service, not operated service.** Beyond the absence of ridership, the file is also a plan and not a record:
    cancellations, breakdowns, driver shortages, road closures and real-time diversions are all invisible. Stage 19's time windows
    are built from *scheduled* departure times and stage 20's window graphs inherit this, so a window whose service is chronically
    not delivered looks the same here as one that ran perfectly. Nothing in the project is validated against operated or real-time
    data (GTFS-RT).
14. **The demand weight is a population proxy with its own parameters.** Stage 21's weighting depends on a catchment radius, a
    distance-decay length and a service exponent, all chosen rather than estimated; the stage runs a sensitivity check, but the
    ranking it produces is conditional on these choices. It also inherits every gap in the CBS link: stations falling outside a
    statistical area, and areas with no recorded population, contribute zero weight, so the demand-weighted rankings of those
    stations are floors and not estimates.
15. **The time windows don't overlap and are coarse.** Stage 19 cuts the day into five blocks. A line running only at the boundary
    of two windows is diluted in both; service interval and waiting time aren't modeled within a window; and the night window spans
    seven hours over which service varies enormously. Comparisons between windows are comparisons between these specific cuts.
16. **The rerouting model is sampled, and its candidate set is small.** Stage 22 evaluates only the top few dozen candidate
    stations against a sample of origin-destination pairs, and prices an unreachable pair at a fixed penalty rather than a modeled
    alternative. Its rank correlation against topology is reported with a p-value precisely because the candidate set is too small
    to establish that the relationship differs from zero; the low correlation shouldn't be read as proven independence.
17. **The eight lenses don't cover the same universe.** Some lenses can score every station (betweenness, service volume,
    demand-weighted), while others are defined only on a subset - cut nodes, top-decile stations with no alternative within walking
    reach, the handful of stations the rerouting model evaluated. Stage 23 records each lens's coverage share alongside its
    ranking, and the pairwise agreements should be read with that in mind: low overlap between a full-coverage lens and a narrow
    one is partly due to the coverage difference. Stage 17 wasn't run in this pass, so the multimodal transfer lens is missing
    entirely.


## 28. Future work

Ordered by expected value per unit of effort. Three items from the original list have since been built and now appear as stages
14-23 - the travel-time graph (18), time-dependent graphs and resilience (19-20), and multimodal layer separation (14-16) - so what
follows is what remains, with those items rewritten to describe the step *after* the one already done.

1. **Consolidate platforms into parent stations.** Rebuild the graph on `parent_station` (with a fallback to spatial clustering for
   stations that don't have one) and re-run stages 02-07. It's cheap, and it directly addresses limitation 4 - the one most likely
   to change the criticality rankings today. Reporting both the split and the consolidated ranking would also quantify how large
   the artifact was.
2. **Normalize edge weights to service on a representative day.** Open `calendar.txt` / `calendar_dates.txt` so a weight becomes
   "trips on an average weekday" instead of "trip rows in the file", making weighted degree comparable across agencies (limitation
   3).
3. **Replace the latitude-longitude region rule with official boundaries.** Link stations to municipal and district polygons, as
   stage 09 already does for CBS statistical areas, and re-run the regional comparison.
4. **Make substitutability functional rather than spatial.** Extend stage 05 so an alternative station is counted only if it's
   served by a line that reaches a similar set of destinations at a similar frequency. The isolation share will rise; the
   interesting question is by how much.
5. **Complete the time-dependent model.** Stages 18-20 and 22 did the first three steps - travel-time edge weights, per-window
   graphs, and disruption measured in added seconds rather than component membership. What's still missing is the part that needs a
   true time-expanded graph: waiting time, headway-dependent transfer penalties, and departure-time-dependent routing. Only then
   would "how much worse is the trip" also include the wait, which for off-peak passengers dominates the in-vehicle travel time
   that stage 18 currently models.
6. **Validate against observed demand.** If ridership or smartcard data can be obtained, test whether structural criticality
   predicts realized passenger disruption, and whether stage 21's population proxy is a usable substitute for it. This is the
   experiment that would turn the project's headline from a topological claim into an operational one - and the only thing that
   could invalidate limitations 2 and 14.
7. **Exact or better-bounded betweenness.** Compute exact betweenness on the largest component in a parallel implementation, or use
   a sampling scheme with error guarantees, and report confidence intervals on the criticality ranking instead of a point
   estimate.
8. **Complete the multilayer model.** Stages 14-16 separated the modes into one graph each and stage 15 measured how different the
   merged national graph is from the bus graph alone, but the layers are still analyzed side by side rather than coupled. The
   missing piece is stage 17: explicit transfer edges between modes at shared or walkable stations, with a transfer cost, so that
   failure propagation between layers can be simulated and the multimodal lens can return to the stage 23 comparison.
9. **A defensible intervention experiment.** Instead of adding edges suggested by embedding similarity - which produced no
   measurable improvement - frame edge addition as an explicit optimization problem (maximizing algebraic connectivity, or
   minimizing worst-case component loss, under a budget) and compare against random and degree-based baselines. Report the null
   result honestly if it recurs.
10. **Reconcile the lenses into a decision rule.** Stage 23 measures how little the eight criticality definitions agree, but stops
    at the small agreed-upon list. The next step is a stated objective - for example minimizing the expected lost passenger-seconds
    under a budget of protected stations - against which every lens can be scored, so that "which definition should we use" becomes
    a question that can be answered rather than a choice.
11. **Validate on a second file and another country.** Re-run the pipeline on a later Israeli GTFS version to separate stable
    structure from file noise, and on a comparable national file to test whether the gap between operational and structural
    criticality is a property of this network or of national public-transport networks in general.


## 29. Reproduction

**Run order.** The notebooks form a chain: each reads the earlier stages' folders under `outputs/nb/` and writes only its own. Run
them in numeric order, from `00` to `24`. Stages 00-13 build and analyze the merged national graph; 14-17 split it by transport
mode; 18-20 replace the hop-count and whole-day assumptions with travel time and time windows; 21-22 push toward the passenger; 23
compares every resulting criticality definition; 24 (this notebook) reads them all and writes nothing any other stage consumes.

| # | Notebook | Writes to `outputs/nb/` | Depends on |
|---|---|---|---|
| 00 | `00_setup_and_data` | `00_setup_and_data/` | the raw GTFS folder `israel-public-transportation/` |
| 01 | `01_data_preparation` | `01_data_preparation/` | 00 |
| 02 | `02_graph_construction` | `02_graph_construction/` | 01 (+ raw `stop_times.txt`) |
| 03 | `03_descriptive_analysis` | `03_descriptive_analysis/` | 02 |
| 04 | `04_centrality_analysis` | `04_centrality_analysis/` | 02 |
| 05 | `05_critical_station_isolation` | `05_critical_station_isolation/` | 04 |
| 06 | `06_robustness_analysis` | `06_robustness_analysis/` | 02, 03, 04 |
| 07 | `07_regional_comparison` | `07_regional_comparison/` | 03, 04 |
| 08 | `08_community_detection` | `08_community_detection/` | 02 |
| 09 | `09_socioeconomic_equity` | `09_socioeconomic_equity/` | 04, 08 (+ CBS network service) |
| 10 | `10_network_model_comparison` | `10_network_model_comparison/` | 02 |
| 11 | `11_rail_network_analysis` | `11_rail_network_analysis/` | the raw GTFS folder |
| 12 | `12_rail_socioeconomic` | `12_rail_socioeconomic/` | 11 (+ CBS network service) |
| 13 | `13_embeddings_link_prediction` | `13_embeddings_link_prediction/` | 02 |
| 14 | `14_multimodal_inventory` | `14_multimodal_inventory/` | the raw GTFS folder (streams `stop_times.txt`) |
| 15 | `15_bus_network` | `15_bus_network/` | the raw GTFS folder (+ 14 for the mode labels) |
| 16 | `16_lightrail_and_minor_modes` | `16_lightrail_and_minor_modes/` | the raw GTFS folder (+ 14) |
| 17 | `17_multimodal_transfer_hubs` | `17_multimodal_transfer_hubs/` | 14, 15, 16 - **optional**; skipping it only removes the multimodal lens from 23 |
| 18 | `18_travel_time_network` | `18_travel_time_network/` | 02, 04 (+ raw `stop_times.txt`) |
| 19 | `19_time_of_day_graphs` | `19_time_of_day_graphs/` | the raw GTFS folder (+ 02) |
| 20 | `20_dynamic_resilience` | `20_dynamic_resilience/` | 19 (+ 04 for the static baseline) |
| 21 | `21_demand_weighted_criticality` | `21_demand_weighted_criticality/` | 02, 04, 08 (CBS population) |
| 22 | `22_rerouting_model` | `22_rerouting_model/` | 18 (travel-time edges), 04 (candidate stations) |
| 23 | `23_critical_station_lenses` | `23_critical_station_lenses/` | 02, 03, 04, 05, 20, 21, 22 (+ 17 if run) |
| 24 | `24_conclusions` (this notebook) | `24_conclusions/` | all of the above, optionally |

**Chain properties.**

* **Stage isolation.** Each notebook writes to exactly one folder named after it, and reads earlier stages read-only. Re-running one
  stage never corrupts another. The old folders `outputs/tables`, `outputs/figures` and `outputs/rail` are read-only across the
  whole series.
* **Self-contained notebooks.** None of the notebooks import from `src/` or `public_transport_network_research/`; each carries the
  code it needs. Deleting those folders doesn't break the series.
* **Determinism.** The random seeds are fixed in every stage that samples (betweenness sampling and its stability check in 04, the
  random-failure trials in 06, the null models in 10, the train/test split and node2vec walks in 13, the pivot samples in 18 and
  20, the demand and uniform-control seeds in 21, the origin-destination sample in 22). Re-running reproduces the same numbers on
  the same file; changing a seed slightly changes the sampled betweenness and so moves the tail of the criticality ranking. Stages
  18 and 20 both quantify this seed sensitivity explicitly as a noise floor within the same measure, which is what makes their
  comparisons across measures and across windows interpretable.
* **Partial runs supported.** This notebook's availability audit in section 5 reports which stages exist, and every section degrades
  to a printed message rather than an error. The headline table simply contains fewer rows.
* **Environment.** Each notebook installs only missing packages via `_ensure(...)`, so both a clean Colab environment and a local
  copy work. The heavy inputs (`stop_times.txt`, ~800 MB) are needed by stages 00, 02, 04, 10, 11, 13, 14, 15, 16, 18 and 19, all
  of which stream the file instead of loading it; stages 03, 05-09, 12, 17, 20, 21, 22, 23 and 24 all run on the saved CSV/JSON
  outputs of earlier stages.
* **Run time.** The expensive stages are 02 (parsing `stop_times.txt`), 04 (sampled betweenness), 06 (repeated component
  computations over the removal sequence), 13 (node2vec), 14/15/16/18/19 (each streams the full `stop_times.txt` once) and 20
  (betweenness and attack simulation per time window). This notebook is effectively instant - it only reads tables and draws two
  figures.


## Conclusions

* **The most useful sentence in this project:** the stations an operator monitors (service volume) and the stations that hold the
  network together (betweenness, cut nodes) are mostly different stations, and the overlap between the two top-K lists is small at
  every depth measured. A resilience plan that starts from the list of busiest stations starts from the wrong list.
* **The fragility is real but must be stated carefully.** Targeted removal damages the network much faster than random failure, yet
  under the corrected normalization - the largest component relative to the stations that still exist - the damage from removing a
  realistic number of stations is far less apocalyptic than the original, wrongly normalized curve implied. Both are printed above;
  the corrected one is the one to cite.
* **Most critical stations have no alternative within walking distance - and this as an upper bound on the good news.** The spatial
  test flags a large share of the top-decile betweenness stations as isolated, and since the test ignores whether the neighboring
  stations reach a useful destination, the true figure can only be worse.
* **Equity has no single-number answer in this data.** The national and within-city correlations disagree in sign, so the chosen
  aggregation level determines the headline. Report the paradox, not one side of it.
* **Rail is the most structurally fragile part of the system and the least statistically knowable.** Almost a path graph, so cut
  nodes are the norm and not the exception - but with only a few dozen stations, its equity tests don't reach significance and
  shouldn't be cited as evidence of fairness.
* **Four results are negative, and they're reported as negative.** The link-prediction advantage was mostly leakage; the suggested
  edges improved simulated resilience by effectively zero; the embedding classifier for critical stations barely beat base-rate
  guessing; and no rail socioeconomic correlation survived correction. None of them is presented as a partial success.
* **There's no one "public transport network" here.** Six modes, six graphs: a national bus fabric where almost every station is
  reachable from every other, and a light-rail graph of disconnected, cycle-free lines where almost every station is a cut node.
  Merging them into one average describes neither.
* **Hop count was the wrong distance measure, and the project now says by how much.** With scheduled travel time on the edges,
  hop-optimal paths almost never match time-optimal ones and cost a large median time penalty; the two betweenness rankings
  disagree far beyond the sampling noise floor. Sections 8-17 answer a hop-count question, and that's how they should be cited.
* **Criticality also depends on the hour.** The night network is substantially smaller, thousands of stations have no service at all
  outside peak hours, and the criticality ranking moves between windows more than the re-sampling noise within the same window.
* **Weighting by people changes the answer - and it's still not ridership.** Stage 21 is CBS population in a walking catchment
  weighted by scheduled service. It moves the ranking meaningfully against a uniform control. It observes no passenger, and must
  never be cited as ridership.
* **Closures are easy to route around, and betweenness doesn't predict their cost.** The median detours are small and almost
  nothing becomes unreachable, but the ranking of stations by passenger-time cost barely correlates with the topological ranking,
  at a p-value consistent with no relationship.
* **The project's central claim, quantified: "critical" is a property of a station *and of a definition*.** Across eight lenses, the
  average pairwise rank correlation is low, some pairs are negatively correlated, structural and operational criticality share
  almost nothing in their top-50 lists, and most flagged stations are flagged by exactly one lens. Choosing a definition is
  therefore a planning decision, not a technical detail.
* **All of the above is conditional on a single GTFS snapshot, sampled betweenness, split platforms and scheduled - not observed -
  service.** The limitations section details the specific ways each of these could move a number, and the future-work section
  orders the fixes by how much they would move it.
